## 0. One-time setup

In [ ]:
#Run this cell once before anything else in the notebook

#Google Drive mount
from google.colab import drive, userdata
drive.mount('/content/drive')

##
from zhipuai import ZhipuAI #in case it is not yet imported
#Standard library
import os
import json
import time
import io

#Data
import numpy as np
import pandas as pd

#Geospatial
import geopandas as gpd
import requests
from shapely.geometry import Point
from pyproj import Transformer

#Spatial statistics
from libpysal.weights import KNN, lag_spatial
from esda.moran import Moran, Moran_Local

#Machine learning / evaluation
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

#Visualisation
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from scipy.stats import norm

#LLM clients
from openai import OpenAI
from huggingface_hub import InferenceClient

#Install any missing packages (uncomment on first run)
!pip install openai groq huggingface_hub geopandas libpysal esda contextily -q
!pip install ZhipuAI #to start using GLM AI model
#path helper
BASE_DIR = '/content/drive/MyDrive/thesis/'   # < edit here if needed

def path(filename: str) -> str:
    """Return the full Drive path for a thesis file."""
    return BASE_DIR + filename

#LLM client connections
client_gpt = OpenAI(api_key=userdata.get('KEY_2'))
client_llama_hf = InferenceClient(api_key=userdata.get('Key_huggingface'))
client_glm = InferenceClient(api_key=userdata.get('GLM'))

models = [
    ("glm",          client_glm, "glm",                        200, "max_tokens"),
    ("llama-3.3-70b-versatile",client_llama_hf, "meta-llama/Llama-3.3-70B-Instruct",   200, "max_tokens"),
    ("gpt-5.1",                client_gpt,      "gpt-5.1",                             200, "max_completion_tokens")]

print("All imports loaded")

## 1. Starting with data load



In [ ]:
#connect drive to work environment
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#load data file to environment
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/thesis/EVS_WVS_Joint_Csv_v5_0.csv')
df.head()

## 2.a Data pre-processing

In [ ]:
# Copy df first
df2 = df.copy()

# Country code mappings from code book reference
east_codes = {"8","51","31","112","70","100","191","203","233","268","348","428","440","498","912","807","616","642","643","911","703","705","804"}
west_codes = {"40","208","246","250","276","300","352","380","528","578","620","724","752","756","826"}

# Create function that identifies east_west from classification of cntry column
def classify(code):
    c = str(int(code)) if pd.notna(code) else ""
    if c in east_codes:
        return 1
    elif c in west_codes:
        return 0
    else:
        return None

df2["east_west"] = df2["cntry"].apply(classify)

# Drop rows where east_west is empty
df2 = df2.dropna(subset=["east_west"])
df2["east_west"] = df2["east_west"].astype(int)

# Count and percentage
# east/west distribution
counts = df2["east_west"].value_counts()
pct = df2["east_west"].value_counts(normalize=True) * 100
total = df2.shape[0]

summary = pd.DataFrame({
    "Label": {1: "East", 0: "West"},
    "Count": counts,
    "Percentage": pct.round(2)
})
print(summary)
print("Total:", total)

# rows per country
country_counts = df2.groupby(["east_west", "cntry"]).size().reset_index(name="count")
country_counts["group"] = country_counts["east_west"].map({1: "East", 0: "West"})
print("\ncountry representation:")
print(country_counts[["group", "cntry", "count"]].sort_values(["group", "count"], ascending=[True, False]).to_string(index=False))

# E-W country count
east_countries = country_counts[country_counts["east_west"] == 1]["cntry"].nunique()
west_countries = country_counts[country_counts["east_west"] == 0]["cntry"].nunique()

print("East countries:", east_countries)
print("West countries:", west_countries)

# wave/year coverage per group
print("\nyear coverage:")
print(df2.groupby("east_west")["year"].agg(["min", "max", "nunique"]).rename(index={1: "East", 0: "West"}).rename(columns={"min": "first year", "max": "last year", "nunique": "unique years"}))

# year breakdown to spot gaps
print("\nunique years per group:")
for label, group in df2.groupby("east_west"):
    name = "East" if label == 1 else "West"
    years = sorted(group["year"].dropna().unique().astype(int).tolist())
    print(f"  {name}: {years}")

# save only East-West respondents dataset (will further on be useful to stratify some observation by country)
df2.to_csv("/content/drive/MyDrive/thesis/east_west_dataset.csv", index=False)

We see that the split is realtively even, thus there will be no need for oversampling/calss weighting. Total number of rows is solid for model training, in total 35 European countries are participating, East and West participation is visible across years, so the dataset is overall appropriate for the further analysis.

In [ ]:
df2.head()
print(df2.dtypes)

Now, let's select variables that contribute to exploring beleifs of Eastern and WEstern-European representatives and explore their missingness pattern (selection is based on variable codebook interpretation and supporting xlsx file):

In [ ]:
# select all relevant variables for research (focus is on 6 general areas selected for research purposes:
#religion, family, political legitimacy, interpersonal and institutional trust, authority and autonomy, and national identity.)
cols = [
    "uniqid", "cntry", "year", "lnge_num", "pwght", "reg_nuts2",
    "A001", "A002", "A003", "A004", "A005", "A006",
    "A027", "A029", "A030", "A032", "A034", "A035", "A038", "A039", "A040",
    "A041", "A042", "A124_02", "A124_03", "A124_06",
    "A124_08", "A124_09", "D081", "D026_03", "D026_05", "D054", "D059",
    "D060", "D061", "D078", "E003", "E004", "E025", "E026", "E027", "E028",
    "E069_01", "E069_04", "E069_05", "E069_06", "E069_08", "E069_12",
    "E069_13", "E069_14", "E069_17", "E111_01", "E114", "E115", "E117",
    "E179_WVS7", "E181_EVS5", "E224", "E225", "E226", "E227", "E228",
    "E229", "E233", "E233A", "E233B", "E235", "E236",
    "E265_01", "E265_02", "E265_03", "E265_04", "E265_05", "E265_06",
    "E265_07", "E265_08", "F025", "F028", "F028B_WVS7", "F066_EVS5",
    "F034", "F050", "F051", "F053", "F054", "F063", "F114A", "F115",
    "F116", "F117", "F118", "F119", "F120", "F121", "F122", "F123",
    "F132", "F144_02", "E290", "A165", "D001_B", "G007_18_B", "G007_33_B",
    "G007_34_B", "G007_35_B", "G007_36_B", "G052", "X001", "X002", "X003",
    "G027A", "X002_02A", "V002", "V001", "X007", "X026_01", "X011", "X013",
    "X025A_01", "X025R", "X028", "W003", "X052", "X035_EVS5",
    "X047_WVS7", "X047E_EVS5", "east_west"
]
print(len(cols))
# keep only columns that exist in df2
cols = [c for c in cols if c in df2.columns]
df2 = df2[cols].copy()

# replace negative values with NaN
num_cols = df2.select_dtypes(include="number").columns
df2[num_cols] = df2[num_cols].where(df2[num_cols] >= 0)

# handle reg_nuts2 separately - it's stored as string but may contain negative codes
df2["reg_nuts2"] = df2["reg_nuts2"].apply(
    lambda x: None if str(x).strip().lstrip('-').isdigit() and float(x) < 0 else x
)

# missing data overview
missing = df2.isnull().sum()
pct_missing = (missing / len(df2) * 100).round(2)

missing_overview = pd.DataFrame({
    "missing": missing,
    "complete": len(df2) - missing,
    "% missing": pct_missing
}).sort_values("% missing", ascending=False)

print(missing_overview.to_string())
print()

We see that nine variables exceed the 20% missingness threshold. Political party of one's preferance, income scales, frequency of praying, spouse employment status, and political party appealing are not significantly impacting data incompleteness, because we also have similar question that interpret other political/work choices, so these variables can be dropped. For occupation variables, however, we keep it as-is, with missing values set to zero, implying no occupation of a person.

In [ ]:
# drop high missingness variables
drop_cols = ["E179_WVS7", "X047_WVS7", "F028B_WVS7", "W003", "E181_EVS5", "X047E_EVS5", "F066_EVS5"]
df2 = df2.drop(columns=[c for c in drop_cols if c in df2.columns])

# occupation variables: keep as-is but treat missing as "no occupation"
occ_cols = [c for c in ["X028", "X035_EVS5"] if c in df2.columns]
df2[occ_cols] = df2[occ_cols].fillna(0)

# updated missing overview
missing = df2.isnull().sum()
pct_missing = (missing / len(df2) * 100).round(2)

missing_overview = pd.DataFrame({
    "missing": missing,
    "complete": len(df2) - missing,
    "% missing": pct_missing
}).sort_values("% missing", ascending=False)

print(missing_overview.to_string())
print(f"\nTotal columns: {len(df2.columns)}")
print(f"Total rows: {len(df2)}")

In [ ]:
#now, let's imputate all missing inputs with `NA` to ensure feasible LLM prompting later on
df2 = df2.fillna("NA")

# verify no missing values remain
assert df2.isnull().sum().sum() == 0, "There are still missing values"
print(f"All missing values filled. Dataset shape: {df2.shape}")
print(df2.head())



In [ ]:
#as a final step, we select only those features that do not disclose the respondent's identity (e.g. religion, language, country etc.)
identity_leaking_cols = [
    "cntry",        # Country ISO numeric code
    "lnge_num",     # Language of interview
    "F025",         # Religious denomination: Orthodox would mean East, Catholic/Protestant- West
    "G027A",        # Respondent immigrant / born in country
    "X002_02A",     # Respondent country of birth (ISO numeric)
    "V002",         # Mother immigrant / born in country
    "V001",         # Father immigrant / born in country
]

#Drop only columns that exist in the dataframe
cols_to_drop = [c for c in identity_leaking_cols if c in df2.columns]
df2 = df2.drop(columns=cols_to_drop)

print(f"Remaining columns: {len(df2.columns)}")
print(f"Dataset shape: {df2.shape}")
# save to Google Drive
output_path = "/content/drive/MyDrive/thesis/clean_dataset.csv"
df2.to_csv(output_path, index=False)
print(f"Dataset saved to: {output_path}")

## 2.b Prompt cleaning

To ensure clean and efficient prompting for LLM, here are several beneficial features we will use. First, we arrange a fixed order of same-output questions that follow an identical structure but reveal different clutural beliefs. This allows to recognize the repeated prefix and avoid reprocessing, saving approximately 4,000 tokens per query. Second, we will transform each response in a compact semicolon-separated format, following the exact attribute order defined in the system prompt. This keeps the variable part of the prompt as short as possible. Third, missing values are already encoded as NA in step `2.a` rather
than skipped, preventing the chain of features to break. Lastly, any attributes that could reveal the geographic or national identity of
the respondent were excluded from the prompt to prevent the model from relying on direct identity markers (step `2.a`).


In [ ]:
import pandas as pd
path = lambda f: f"/content/drive/MyDrive/thesis/{f}"
groups = [
    # IDENTIFIERS
    ("uniqid",                    ["uniqid"],                                                              "id"),
    ("year",                      ["year"],                                                                "id"),
    ("pwght",                     ["pwght"],                                                               "id"),

    # IMPORTANCE IN LIFE: family; friends; leisure; politics; work; religion
    ("importance_life",           ["A001","A002","A003","A004","A005","A006"],                            "section"),

    # CHILD QUALITIES: manners; independence; hard work; responsibility; imagination;
    # tolerance; thrift; determination; religious faith; unselfishness; obedience
    ("child_qualities",           ["A027","A029","A030","A032","A034",
                                   "A035","A038","A039","A040","A041","A042"],                            "section"),

    # TOLERANCE NEIGHBOURS: different race; heavy drinkers; immigrants; drug addicts; homosexuals
    ("tolerance",                 ["A124_02","A124_03","A124_06","A124_08","A124_09"],                   "section"),

    # FAMILY AND GENDER: gay parents; duty children; care ill parent; make parents proud;
    # men better leaders; university boy; working mother; men better executives
    ("family_gender",             ["D081","D026_03","D026_05","D054",
                                   "D059","D060","D061","D078"],                                          "section"),

    # POLITICAL LEGITIMACY AIMS: first choice; second choice
    ("aims",                      ["E003","E004"],                                                        "section"),

    # POLITICAL ACTION: petition; boycotts; demonstrations; unofficial strikes
    ("political_action",          ["E025","E026","E027","E028"],                                         "section"),

    # INSTITUTIONAL CONFIDENCE: churches; press; unions; police; civil services;
    # political parties; companies; environment movement; courts
    ("confidence",                ["E069_01","E069_04","E069_05","E069_06","E069_08",
                                   "E069_12","E069_13","E069_14","E069_17"],                              "section"),

    # POLITICAL SYSTEM SATISFACTION AND PREFERENCES
    ("political_system_satisfaction", ["E111_01"],                                                        "single"),
    ("political_system_pref",     ["E114","E115","E117"],                                                 "section"),

    # DEMOCRACY PERCEPTIONS: tax rich; religious laws; free elections; unemployment aid;
    # army takeover; civil rights; women rights; equal incomes; obey rulers
    ("democracy_features",        ["E224","E225","E226","E227","E228",
                                   "E229","E233","E233A","E233B"],                                        "section"),
    ("democracy_importance",      ["E235"],                                                                "single"),
    ("democracy_current",         ["E236"],                                                                "single"),

    # ELECTION QUALITY: fair count; opposition blocked; TV bias; bribery;
    # fair journalism; fair officials; rich buy elections; violence
    ("election_quality",          ["E265_01","E265_02","E265_03","E265_04",
                                   "E265_05","E265_06","E265_07","E265_08"],                             "section"),

    # RELIGION: attendance; identity; beliefs; importance of god
    ("religious_attendance",      ["F028"],                                                                "single"),
    ("religious_identity",        ["F034"],                                                                "single"),
    ("religious_beliefs",         ["F050","F051","F053","F054"],                                         "section"),
    ("god_importance",            ["F063"],                                                                "single"),

    # JUSTIFIABLE BEHAVIORS: benefits fraud; fare dodging; tax cheat; bribe; homosexuality;
    # prostitution; abortion; divorce; euthanasia; suicide; casual sex; death penalty; political violence
    ("justifiable",               ["F114A","F115","F116","F117","F118","F119",
                                   "F120","F121","F122","F123","F132",
                                   "F144_02","E290"],                                                     "section"),

    # TRUST: general; family; neighbourhood; people you know; strangers;
    # another religion; another nationality
    ("general_trust",             ["A165"],                                                                "single"),
    ("trust_groups",              ["D001_B","G007_18_B","G007_33_B",
                                   "G007_34_B","G007_35_B","G007_36_B"],                                 "section"),

    # IMMIGRATION IMPACT
    ("immigrant_impact",          ["G052"],                                                                "single"),

    # DEMOGRAPHICS: sex; birth year; age; marital status; children; household;
    # education isced; education recoded; employment; occupation sector
    ("sex",                       ["X001"],                                                                "single"),
    ("birth_year",                ["X002"],                                                                "single"),
    ("age",                       ["X003"],                                                                "single"),
    ("marital_status",            ["X007"],                                                                "single"),
    ("n_children",                ["X011"],                                                                "single"),
    ("household_size",            ["X013"],                                                                "single"),
    ("education_isced",           ["X025A_01"],                                                           "single"),
    ("education_r",               ["X025R"],                                                              "single"),
    ("employment",                ["X028"],                                                                "single"),
    ("occupation_sector",         ["X052"],                                                               "single"),

    # TARGET
    ("east_west",                 ["east_west"],                                                          "target"),
]

def restructure(df, groups):
    new_df = pd.DataFrame()
    for new_col, src_cols, kind in groups:
        existing = [c for c in src_cols if c in df.columns]
        if not existing:
            continue
        if len(existing) == 1:
            new_df[new_col] = df[existing[0]]
        else:
            new_df[new_col] = df[existing].apply(
                lambda row: ";".join(
                    "NA" if pd.isna(v)
                    else str(int(v)) if isinstance(v, float) and v.is_integer()
                    else str(v)
                    for v in row
                ),
                axis=1
            )
    return new_df

# Load and restructure
df_raw        = pd.read_csv(path("clean_dataset.csv"))
df_structured = restructure(df_raw, groups)

# Save
df_structured.to_csv(path("structured_dataset.csv"), index=False)

print(f"Shape:   {df_structured.shape}")
print(f"Columns: {len(df_structured.columns)}")
print(df_structured.columns.tolist())
print("\nSample row:")
print(df_structured.iloc[37]) #one respondent as an example

The following restructuring decreased 112 feature columns to 35, which will improve memory usage for test prompting. Nice!

## 3.a LLM comparisons. Test sample and AI connection

to start with model training on the whole dataset, we will first experiment with 100 entries on three different models from diverse ai families: **Gpt-5.1** as the leading closed-source model trained on broadest multilingual corpus, **Llama 3.3 70B** as the open-source alternative trained predominantly on Western-centric data, and, lastly, **Deepseek V3** as a non-Western open-source model whose distinct training allow us to test whether the cultural background of a model influences its classification of European value systems. Overall, we try to see which one can understand and give the highest accuracy output among them all.

In [ ]:
#test dataset with random selector of 100 respondents to train LLM, East-West split should be around 53 to 47 accordingly.
# sample keeping east/west ratio
path = lambda f: f"/content/drive/MyDrive/thesis/{f}"
df2 = pd.read_csv('/content/drive/MyDrive/thesis/structured_dataset.csv')
east_sample = df2[df2["east_west"] == 1].sample(n=53, random_state=1234)
west_sample = df2[df2["east_west"] == 0].sample(n=47, random_state=1234)

test_llm = pd.concat([east_sample, west_sample]).sample(frac=1, random_state=1234).reset_index(drop=True)

print(f"Total: {len(test_llm)}")
print(f"East: {len(test_llm[test_llm['east_west'] == 1])} ({len(test_llm[test_llm['east_west'] == 1])}%)")
print(f"West: {len(test_llm[test_llm['east_west'] == 0])} ({len(test_llm[test_llm['east_west'] == 0])}%)")

# save to drive
test_llm.to_csv('/content/drive/MyDrive/thesis/test_llm_100.csv', index=False)
print("saved to drive/thesis/test_llm_100.csv")

In [ ]:
# test dataset with 500 respondents, East-West split 53/47
#df2 = pd.read_csv(path("structured_dataset.csv"))

#east_n = round(500 * 0.53)  # 265
#west_n = 500 - east_n       # 235

#east_sample = df2[df2["east_west"] == 1].sample(n=east_n, random_state=1234)
#west_sample = df2[df2["east_west"] == 0].sample(n=west_n, random_state=1234)

#test_llm_500 = pd.concat([east_sample, west_sample]).sample(frac=1, random_state=1234).reset_index(drop=True)

#print(f"Total: {len(test_llm_500)}")
#print(f"East:  {len(test_llm_500[test_llm_500['east_west'] == 1])} ({east_n/500*100:.0f}%)")
#print(f"West:  {len(test_llm_500[test_llm_500['east_west'] == 0])} ({west_n/500*100:.0f}%)")

#test_llm_500.to_csv(path("test_llm_500.csv"), index=False)
#print("Saved to thesis/test_llm_500.csv")

In [ ]:
# install openai, groq and mistralai for further LLM usage
!pip install openai -q
!pip install groq
!pip install hf

In [ ]:
#loading key for gpt-5.1 model
#from openai import OpenAI
#from google.colab import userdata
#client_gpt = OpenAI(api_key=userdata.get('KEY_2'))
#print("connected")


In [ ]:
#loading key for Llama 3.3 70B
#not enough tokens for 100 samples, use Hugging face setup instead
#from groq import Groq
#client_llama = Groq(api_key=userdata.get('Key_llama'))
#print("connected")

In [ ]:
#from huggingface_hub import InferenceClient
#client_llama_hf = InferenceClient(api_key=userdata.get('Key_huggingface'))
#print("connected")

In [ ]:
#use the OpenAI-compatible endpoint for Deepseek to work
#loading key for Deepseek
#client_deepseek = OpenAI(
#    api_key=userdata.get('Key_deepseek'),
#    base_url="https://api.deepseek.com"
#)
#print("connected")

In [ ]:
#setup all three clients
models = [
    ("deepseek-chat",   client_deepseek, "deepseek-chat",     200, "max_tokens"),
    ("llama-3.3-70b-versatile",client_llama_hf,   "meta-llama/Llama-3.3-70B-Instruct",  200, "max_tokens"),
    ("gpt-5.1",                client_gpt,     "gpt-5.1",                  200, "max_completion_tokens"),
]

Now it's time to try the models justification with the prompt. Generally, we are interested to see what reasoning they use to distinguish between East and West, so the expected output will be prediced East-West column, if the output was correct for easier filtering, two main domains the model justified its reasoning on, confidence level of response and the reasoning itself

In [ ]:
import pandas as pd
from openai import OpenAI
from google.colab import userdata
# Load data
path = lambda f: f"/content/drive/MyDrive/thesis/{f}"
test_llm_100 = pd.read_csv(path("test_llm_100.csv"))

# SYSTEM PROMPT fixed cached across all respondents for all three models
SYSTEM_PROMPT = """
# ROLE
You are analyzing a European Values Survey respondent. You will be given respondent
values in a fixed order. Each value corresponds to the matching attribute listed below.

# IMPORTANT RULES
- Classify based only on values, beliefs, and attitudes.
- Missing values are encoded as NA. Ignore them when reasoning.
- Values appear in exactly the same order as the listed attributes, separated by ";".
- Do not infer nationality from any single attribute — reason holistically.
- Reason must be under 15 words, facts only, no filler words, no comparisons to East or West.

# ATTRIBUTE DEFINITIONS

[IMPORTANCE IN LIFE]
Scale: 1=very important; 2=rather important; 3=not very important; 4=not important at all
Order: Family; Friends; Leisure time; Politics; Work; Religion

[CHILD QUALITIES]
Scale: 1=mentioned as important; 0=not mentioned
Order: Good manners; Independence; Hard work; Feeling of responsibility; Imagination;
       Tolerance and respect; Thrift; Determination; Religious faith; Unselfishness; Obedience

[TOLERANCE — neighbours]
Scale: 0=would not accept; 1=would accept
Order: People of a different race; Heavy drinkers; Immigrants; Drug addicts; Homosexuals

[FAMILY AND GENDER]
Scale: 1=strongly agree; 2=agree; 3=neither; 4=disagree; 5=strongly disagree
Order: Homosexual couples are as good parents; Duty to have children;
       Child must care for ill parent; Main goal to make parents proud;
       Men make better political leaders; University more important for boys;
       Pre-school child suffers with working mother; Men make better business executives

[POLITICAL LEGITIMACY — AIMS]
Scale: 1=maintaining order; 2=more say in decisions; 3=fighting rising prices; 4=freedom of speech
Order: Aims first choice; Aims second choice

[POLITICAL ACTION]
Scale: 1=have done; 2=might do; 3=would never do
Order: Signing a petition; Joining boycotts; Attending demonstrations; Joining unofficial strikes

[INSTITUTIONAL CONFIDENCE]
Scale: 1=a lot; 2=quite a lot; 3=not very much; 4=not at all
Order: Churches; The press; Labour unions; The police; Civil services;
       Political parties; Major companies; Environmental movement; Justice system/courts

[POLITICAL SYSTEM]
Satisfaction scale: 1=dissatisfied → 10=satisfied
Preference scale: 1=very good; 2=fairly good; 3=fairly bad; 4=very bad
Order: Satisfaction with political system; Strong leader; Experts make decisions; Democratic system

[DEMOCRACY PERCEPTIONS]
Scale: 1=not an essential characteristic → 10=essential characteristic
Importance scale: 1=not important → 10=absolutely important
Order: Governments tax rich and subsidize poor; Religious authorities interpret laws;
       People choose leaders in free elections; State aid for unemployment;
       Army takes over when government incompetent; Civil rights protect liberty;
       Women have same rights as men; State makes incomes equal; People obey rulers;
       Importance of democracy; How democratically governed today (1=not democratic → 10=very democratic)

[ELECTION QUALITY]
Scale: 1=very often; 2=fairly often; 3=not often; 4=not at all often
Order: Votes counted fairly; Opposition candidates prevented; TV favors governing party;
       Voters bribed; Journalists fair; Election officials fair; Rich buy elections;
       Voters threatened with violence

[RELIGION]
Attendance scale: 1=more than once a week; 2=once a week; 3=once a month;
                  4=special holy days; 5=other holy days; 6=once a year; 7=less often; 8=never
Identity: 1=religious; 2=not religious; 3=atheist
Beliefs scale: 0=no; 1=yes
God importance scale: 1=not important at all → 10=absolutely important
Order: Attendance; Religious identity; Believe in God; Believe in life after death;
       Believe in hell; Believe in heaven; Importance of God in life

[JUSTIFIABLE BEHAVIORS]
Scale: 1=never justifiable → 10=always justifiable
Order: Claiming government benefits not entitled to; Avoiding fare on public transport;
       Cheating on taxes; Accepting a bribe; Homosexuality; Prostitution; Abortion;
       Divorce; Euthanasia; Suicide; Casual sex; Death penalty; Political violence

[TRUST]
General trust: 0=cannot be too careful; 1=most people can be trusted
Group trust scale: 1=trust completely; 2=trust somewhat; 3=not very much; 4=not at all
Order: General trust in people; Trust in family; Trust in neighbourhood;
       Trust in people you know; Trust in strangers; Trust in people of another religion;
       Trust in people of another nationality

[IMMIGRATION]
Scale: 1=very bad; 2=quite bad; 3=neither; 4=quite good; 5=very good
Order: Impact of immigrants on country development

[DEMOGRAPHICS]
Order: Sex (1=male; 2=female); Birth year; Age;
       Marital status (1=married; 2=living together; 3=divorced; 4=separated; 5=widowed; 6=single);
       Number of children; Household size;
       Education ISCED (0=less than primary → 8=doctoral);
       Education recoded (1=lower; 2=middle; 3=upper);
       Employment (0,NA=no paid work; 1=full time; 2=part time; 3=self employed;
                   4=retired; 5=housewife; 6=student; 7=unemployed; 8=other);
       Occupation sector (1=public; 2=private; 3=non-profit)

# TASK
Classify whether the respondent is from Eastern or Western Europe based purely on the
provided values, beliefs, and attitudes.

# OUTPUT FORMAT
Respond with exactly this structure and nothing else:
Classification: East or West
Primary domain: [choose one-two and mention them through ";" in one column: religion | family | political legitimacy | trust | authority and autonomy | national identity]
Confidence: [high | medium | low]
Reason: [max 15 words, comma-separated facts only, no adjectives like "very" or "much", no comparisons]
"""

# User prompt builder
def fmt(row, col):
    v = row.get(col, None)
    if pd.isna(v):
        return "NA"
    if isinstance(v, float) and v.is_integer():
        return str(int(v))
    return str(v)

def fmt_group(row, col):
    v = row.get(col, None)
    if pd.isna(v):
        return "NA"
    return str(v)

def row_to_user_prompt(row):
    political_system = ";".join([
        fmt(row, "political_system_satisfaction"),
        fmt_group(row, "political_system_pref")
    ])
    democracy = ";".join([
        fmt_group(row, "democracy_features"),
        fmt(row, "democracy_importance"),
        fmt(row, "democracy_current")
    ])
    religion = ";".join([
        fmt(row, "religious_attendance"),
        fmt(row, "religious_identity"),
        fmt_group(row, "religious_beliefs"),
        fmt(row, "god_importance")
    ])
    trust = ";".join([
        fmt(row, "general_trust"),
        fmt_group(row, "trust_groups")
    ])
    demographics = ";".join([
        fmt(row, "sex"),
        fmt(row, "birth_year"),
        fmt(row, "age"),
        fmt(row, "marital_status"),
        fmt(row, "n_children"),
        fmt(row, "household_size"),
        fmt(row, "education_isced"),
        fmt(row, "education_r"),
        fmt(row, "employment"),
        fmt(row, "occupation_sector")
    ])

    return (
        f"[IMPORTANCE IN LIFE] {fmt_group(row, 'importance_life')}\n"
        f"[CHILD QUALITIES] {fmt_group(row, 'child_qualities')}\n"
        f"[TOLERANCE] {fmt_group(row, 'tolerance')}\n"
        f"[FAMILY AND GENDER] {fmt_group(row, 'family_gender')}\n"
        f"[POLITICAL LEGITIMACY AIMS] {fmt_group(row, 'aims')}\n"
        f"[POLITICAL ACTION] {fmt_group(row, 'political_action')}\n"
        f"[INSTITUTIONAL CONFIDENCE] {fmt_group(row, 'confidence')}\n"
        f"[POLITICAL SYSTEM] {political_system}\n"
        f"[DEMOCRACY PERCEPTIONS] {democracy}\n"
        f"[ELECTION QUALITY] {fmt_group(row, 'election_quality')}\n"
        f"[RELIGION] {religion}\n"
        f"[JUSTIFIABLE BEHAVIORS] {fmt_group(row, 'justifiable')}\n"
        f"[TRUST] {trust}\n"
        f"[IMMIGRATION] {fmt(row, 'immigrant_impact')}\n"
        f"[DEMOGRAPHICS] {demographics}\n"
    )

In [ ]:
#one loop over all three models
import time
for model_name, client, model_id, max_tok, tok_param in models:
    print(f"\n{'='*50}")
    print(f"Running: {model_name}")
    print(f"{'='*50}")
    results = []

    for i, row in test_llm_100.iterrows():
        user_prompt = row_to_user_prompt(row)

        # tok_param handles the difference between OpenAI and Groq param names
        kwargs = {
            "model": model_id,
            "temperature": 0,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_prompt}
            ],
            tok_param: max_tok  # max_completion_tokens for GPT, max_tokens for others
        }

        response = client.chat.completions.create(**kwargs)
        prediction_raw = response.choices[0].message.content.strip()
        actual = "East" if row["east_west"] == 1 else "West"

        classification, domain, confidence, reason = "NA", "NA", "NA", "NA"
        for line in prediction_raw.splitlines():
            line = line.strip()
            if line.startswith("Classification:"):
                classification = line.split(":", 1)[1].strip()
            elif line.startswith("Primary domain:"):
                domain = line.split(":", 1)[1].strip()
            elif line.startswith("Confidence:"):
                confidence = line.split(":", 1)[1].strip()
            elif line.startswith("Reason:"):
                reason = line.split(":", 1)[1].strip()

        correct = classification == actual
        results.append({
            "uniqid":     row.get("uniqid", i),
            "model":      model_name,
            "actual":     actual,
            "predicted":  classification,
            "correct":    correct,
            "domain":     domain,
            "confidence": confidence,
            "reasoning":  reason,
        })

        print(f"[{len(results)}/100] actual={actual} | predicted={classification} | "
              f"confidence={confidence} | correct={correct}")


    # Summary per model
    results_df = pd.DataFrame(results)
    accuracy = results_df["correct"].mean() * 100
    print(f"\nAccuracy: {accuracy:.1f}%")
    print(f"Prediction breakdown:\n{results_df['predicted'].value_counts()}")
    print(f"Domain breakdown:\n{results_df['domain'].value_counts()}")

    # Save individual CSV per model
    safe_name = model_name.replace(".", "-").replace(":", "-")
    results_df.to_csv(path(f"llm_results_100_{safe_name}.csv"), index=False)
    print(f"Saved: llm_results_100_{safe_name}.csv")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import pandas as pd

model_files = [
    ("GPT-5.1",        "llm_results_100_gpt-5-1.csv"),
    ("GLM-5.1",  "llm_results_100_glm-5.csv"),
    ("Llama 3.3 70B",  "llm_results_100_llama-3-3-70b-versatile.csv"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (model_label, filename) in zip(axes, model_files):

    results_df = pd.read_csv(path(filename))
    results_df_clean = results_df[results_df["predicted"].isin(["East", "West"])].copy()

    actual    = results_df_clean["actual"]
    predicted = results_df_clean["predicted"]

    print(f"\n{'='*50}")
    print(f"Model: {model_label}")
    print(f"{'='*50}")
    print(classification_report(actual, predicted, target_names=["East", "West"]))

    cm   = confusion_matrix(actual, predicted, labels=["East", "West"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["East", "West"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"{model_label}")

    print("Domain breakdown:")
    print(results_df_clean["domain"].value_counts().to_string())
    print("\nConfidence breakdown:")
    print(results_df_clean["confidence"].value_counts().to_string())
    print(f"\nRows used:    {len(results_df_clean)}/100")
    print(f"Rows skipped: {len(results_df) - len(results_df_clean)} (invalid prediction)")
    print(f"Accuracy:     {results_df_clean['correct'].mean() * 100:.1f}%")

fig.suptitle("Confusion Matrix Comparison — East/West Classification (n=100)", fontsize=13)
plt.tight_layout()
plt.savefig(path("confusion_matrix_100_all_models.png"), dpi=150)
plt.show()

## 3.b. LLM comparisons. Non-religious prompt

Given that the majority of models' judjements are based on religious domain, a natural concern arises: the model might rely mostly on religious signals as a shortcut to classification. Instead, let us inspect whether the accuracy of model differs were there no instruction about religiousity of a person.

In [ ]:
# SYSTEM PROMPT
SYSTEM_PROMPT = """
# ROLE
You are analyzing a European Values Survey respondent. You will be given respondent
values in a fixed order. Each value corresponds to the matching attribute listed below.

# IMPORTANT RULES
- Classify based only on values, beliefs, and attitudes.
- Missing values are encoded as NA. Ignore them when reasoning.
- Values appear in exactly the same order as the listed attributes, separated by ";".
- The [RELIGION] section is provided but must be completely ignored. Do not use it in your reasoning or classification under any circumstances.
- Do not infer nationality from any single attribute — reason holistically.
- Reason must be under 15 words, facts only, no filler words, no comparisons to East or West.

# ATTRIBUTE DEFINITIONS

[IMPORTANCE IN LIFE]
Scale: 1=very important; 2=rather important; 3=not very important; 4=not important at all
Order: Family; Friends; Leisure time; Politics; Work; Religion

[CHILD QUALITIES]
Scale: 1=mentioned as important; 0=not mentioned
Order: Good manners; Independence; Hard work; Feeling of responsibility; Imagination;
       Tolerance and respect; Thrift; Determination; Religious faith; Unselfishness; Obedience

[TOLERANCE — neighbours]
Scale: 0=would not accept; 1=would accept
Order: People of a different race; Heavy drinkers; Immigrants; Drug addicts; Homosexuals

[FAMILY AND GENDER]
Scale: 1=strongly agree; 2=agree; 3=neither; 4=disagree; 5=strongly disagree
Order: Homosexual couples are as good parents; Duty to have children;
       Child must care for ill parent; Main goal to make parents proud;
       Men make better political leaders; University more important for boys;
       Pre-school child suffers with working mother; Men make better business executives

[POLITICAL LEGITIMACY: AIMS]
Scale: 1=maintaining order; 2=more say in decisions; 3=fighting rising prices; 4=freedom of speech
Order: Aims first choice; Aims second choice

[POLITICAL ACTION]
Scale: 1=have done; 2=might do; 3=would never do
Order: Signing a petition; Joining boycotts; Attending demonstrations; Joining unofficial strikes

[INSTITUTIONAL CONFIDENCE]
Scale: 1=a lot; 2=quite a lot; 3=not very much; 4=not at all
Order: Churches; The press; Labour unions; The police; Civil services;
       Political parties; Major companies; Environmental movement; Justice system/courts

[POLITICAL SYSTEM]
Satisfaction scale: 1=dissatisfied → 10=satisfied
Preference scale: 1=very good; 2=fairly good; 3=fairly bad; 4=very bad
Order: Satisfaction with political system; Strong leader; Experts make decisions; Democratic system

[DEMOCRACY PERCEPTIONS]
Scale: 1=not an essential characteristic → 10=essential characteristic
Importance scale: 1=not important → 10=absolutely important
Order: Governments tax rich and subsidize poor; Religious authorities interpret laws;
       People choose leaders in free elections; State aid for unemployment;
       Army takes over when government incompetent; Civil rights protect liberty;
       Women have same rights as men; State makes incomes equal; People obey rulers;
       Importance of democracy; How democratically governed today (1=not democratic → 10=very democratic)

[ELECTION QUALITY]
Scale: 1=very often; 2=fairly often; 3=not often; 4=not at all often
Order: Votes counted fairly; Opposition candidates prevented; TV favors governing party;
       Voters bribed; Journalists fair; Election officials fair; Rich buy elections;
       Voters threatened with violence

[RELIGION]
Attendance scale: 1=more than once a week; 2=once a week; 3=once a month;
                  4=special holy days; 5=other holy days; 6=once a year; 7=less often; 8=never
Identity: 1=religious; 2=not religious; 3=atheist
Beliefs scale: 0=no; 1=yes
God importance scale: 1=not important at all → 10=absolutely important
Order: Attendance; Religious identity; Believe in God; Believe in life after death;
       Believe in hell; Believe in heaven; Importance of God in life

[JUSTIFIABLE BEHAVIORS]
Scale: 1=never justifiable → 10=always justifiable
Order: Claiming government benefits not entitled to; Avoiding fare on public transport;
       Cheating on taxes; Accepting a bribe; Homosexuality; Prostitution; Abortion;
       Divorce; Euthanasia; Suicide; Casual sex; Death penalty; Political violence

[TRUST]
General trust: 0=cannot be too careful; 1=most people can be trusted
Group trust scale: 1=trust completely; 2=trust somewhat; 3=not very much; 4=not at all
Order: General trust in people; Trust in family; Trust in neighbourhood;
       Trust in people you know; Trust in strangers; Trust in people of another religion;
       Trust in people of another nationality

[IMMIGRATION]
Scale: 1=very bad; 2=quite bad; 3=neither; 4=quite good; 5=very good
Order: Impact of immigrants on country development

[DEMOGRAPHICS]
Order: Sex (1=male; 2=female); Birth year; Age;
       Marital status (1=married; 2=living together; 3=divorced; 4=separated; 5=widowed; 6=single);
       Number of children; Household size;
       Education ISCED (0=less than primary → 8=doctoral);
       Education recoded (1=lower; 2=middle; 3=upper);
       Employment (0,NA=no paid work; 1=full time; 2=part time; 3=self employed;
                   4=retired; 5=housewife; 6=student; 7=unemployed; 8=other);
       Occupation sector (1=public; 2=private; 3=non-profit)

# TASK
Classify whether the respondent is from Eastern or Western Europe based only on the
provided values, beliefs, and attitudes.

# OUTPUT FORMAT
Respond with exactly this structure and nothing else:
Classification: East or West
Primary domain: [choose one-two and mention them through ";" in one column: family | political legitimacy | trust | authority and autonomy | national identity]
Confidence: [high | medium | low]
Reason: [max 15 words, comma-separated facts only, no adjectives like "very" or "much", no comparisons]
"""

In [ ]:
#one loop over all three models for non-religiousity case
for model_name, client, model_id, max_tok, tok_param in models:
    print(f"\n{'='*50}")
    print(f"Running: {model_name}")
    print(f"{'='*50}")
    results = []

    for i, row in test_llm_100.iterrows():
        user_prompt = row_to_user_prompt(row)

        # tok_param handles the difference between OpenAI and Groq param names
        kwargs = {
            "model": model_id,
            "temperature": 0,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": user_prompt}
            ],
            tok_param: max_tok  # max_completion_tokens for GPT, max_tokens for others
        }

        response = client.chat.completions.create(**kwargs)
        prediction_raw = response.choices[0].message.content.strip()
        actual = "East" if row["east_west"] == 1 else "West"

        classification, domain, confidence, reason = "NA", "NA", "NA", "NA"
        for line in prediction_raw.splitlines():
            line = line.strip()
            if line.startswith("Classification:"):
                classification = line.split(":", 1)[1].strip()
            elif line.startswith("Primary domain:"):
                domain = line.split(":", 1)[1].strip()
            elif line.startswith("Confidence:"):
                confidence = line.split(":", 1)[1].strip()
            elif line.startswith("Reason:"):
                reason = line.split(":", 1)[1].strip()

        correct = classification == actual
        results.append({
            "uniqid":     row.get("uniqid", i),
            "model":      model_name,
            "actual":     actual,
            "predicted":  classification,
            "correct":    correct,
            "domain":     domain,
            "confidence": confidence,
            "reasoning":  reason,
        })

        print(f"[{len(results)}/100] actual={actual} | predicted={classification} | "
              f"confidence={confidence} | correct={correct}")


    # Summary per model
    results_df = pd.DataFrame(results)
    accuracy = results_df["correct"].mean() * 100
    print(f"\nAccuracy: {accuracy:.1f}%")
    print(f"Prediction breakdown:\n{results_df['predicted'].value_counts()}")
    print(f"Domain breakdown:\n{results_df['domain'].value_counts()}")

    # Save individual CSV per model
    safe_name = model_name.replace(".", "-").replace(":", "-")
    results_df.to_csv(path(f"llm_results_100_no_religion_{safe_name}.csv"), index=False)
    print(f"Saved: llm_results_100_no_religion_{safe_name}.csv")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import pandas as pd

model_files = [
    ("GPT-5.1",        "llm_results_100_no_religion_gpt-5-1.csv"),
    ("GLM-5.1",  "llm_results_100_glm-5_no_religion.csv"),
    ("Llama 3.3 70B",  "llm_results_100_no_religion_llama-3-3-70b-versatile.csv"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (model_label, filename) in zip(axes, model_files):

    results_df = pd.read_csv(path(filename))
    results_df_clean = results_df[results_df["predicted"].isin(["East", "West"])].copy()

    actual    = results_df_clean["actual"]
    predicted = results_df_clean["predicted"]

    print(f"\n{'='*50}")
    print(f"Model: {model_label}")
    print(f"{'='*50}")
    print(classification_report(actual, predicted, target_names=["East", "West"]))

    cm   = confusion_matrix(actual, predicted, labels=["East", "West"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["East", "West"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"{model_label}")

    print("Domain breakdown:")
    print(results_df_clean["domain"].value_counts().to_string())
    print("\nConfidence breakdown:")
    print(results_df_clean["confidence"].value_counts().to_string())
    print(f"\nRows used:    {len(results_df_clean)}/100")
    print(f"Rows skipped: {len(results_df) - len(results_df_clean)} (invalid prediction)")
    print(f"Accuracy:     {results_df_clean['correct'].mean() * 100:.1f}%")

fig.suptitle("Confusion Matrix Comparison — East/West Classification (n=100)", fontsize=13)
plt.tight_layout()
plt.savefig(path("confusion_matrix_100_all_models_no_religion.png"), dpi=150)
plt.show()

## 4.a Startified dataset. Religious prompt

to be able to track the error of LLM output by region, a sample of 500 per country giving total of 17500 observations for the whole dataset was deemed sufficient for the purposes of this study. This exceeds the statistically established minimum of 384 observations required to estimate a proportion at 95% confidence with a 5% margin of error (Cochran, 1977), while also providing enough misclassified cases per country to support meaningful error analysis.

In [ ]:
import pandas as pd

# Load datasets
structured_dataset = pd.read_csv("/content/drive/MyDrive/thesis/structured_dataset.csv")
east_west_dataset = pd.read_csv("/content/drive/MyDrive/thesis/east_west_dataset.csv")

# Merge structured_dataset with country codes from east_west_dataset
merged = structured_dataset.merge(
    east_west_dataset[["uniqid", "cntry"]],
    on="uniqid",
    how="left"
)

# Sample 500 observations per country
sampled = (
    merged
    .groupby("cntry", group_keys=False)
    .apply(lambda x: x.sample(n=min(500, len(x)), random_state=42))
    .reset_index(drop=True)
)

# Verify total count
total = len(sampled)
per_country = sampled.groupby("cntry").size().reset_index(name="count")

print(f"Total observations selected: {total}")
print(f"Expected: {len(per_country['cntry'].unique()) * 500}")
print("\nObservations per country:")
print(per_country.to_string(index=False))

# Drop cntry and any columns that came from original_dataset, keep original structured_dataset columns only
original_cols = structured_dataset.columns.tolist()
structured_dataset_stratified = sampled[original_cols].reset_index(drop=True)

print(f"\nFinal dataset shape: {structured_dataset_stratified.shape}")
print(f"Columns: {structured_dataset_stratified.columns.tolist()}")
# Save to CSV
structured_dataset_stratified.to_csv("/content/drive/MyDrive/thesis/structured_dataset_stratified.csv", index=False)
print("Saved to structured_dataset_stratified.csv")

Before starting the prompt, we also define path to imeediately populate all the batches into the thesis folder. It also saves time to every time impute the save line, as well as saving each batch immediately as soon as it is generated

In [ ]:
import pandas as pd
import json
import time
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Paths
BASE_DIR = '/content/drive/MyDrive/thesis/' #change your path for automatic save
def path(f): return BASE_DIR + f

# Load full dataset
structured_dataset = pd.read_csv(BASE_DIR + 'structured_dataset_stratified.csv')
print(f'Total respondents: {len(structured_dataset)}')

In [ ]:
# SYSTEM PROMPT
SYSTEM_PROMPT = """
# ROLE
You are analyzing a European Values Survey respondent. You will be given respondent
values in a fixed order. Each value corresponds to the matching attribute listed below.

# IMPORTANT RULES
- Classify based only on values, beliefs, and attitudes.
- Missing values are encoded as NA. Ignore them when reasoning.
- Values appear in exactly the same order as the listed attributes, separated by ";".
- Do not infer nationality from any single attribute — reason holistically.
- Reason must be under 15 words, facts only, no filler words, no comparisons to East or West.

# ATTRIBUTE DEFINITIONS

[IMPORTANCE IN LIFE]
Scale: 1=very important; 2=rather important; 3=not very important; 4=not important at all
Order: Family; Friends; Leisure time; Politics; Work; Religion

[CHILD QUALITIES]
Scale: 1=mentioned as important; 0=not mentioned
Order: Good manners; Independence; Hard work; Feeling of responsibility; Imagination;
       Tolerance and respect; Thrift; Determination; Religious faith; Unselfishness; Obedience

[TOLERANCE — neighbours]
Scale: 0=would not accept; 1=would accept
Order: People of a different race; Heavy drinkers; Immigrants; Drug addicts; Homosexuals

[FAMILY AND GENDER]
Scale: 1=strongly agree; 2=agree; 3=neither; 4=disagree; 5=strongly disagree
Order: Homosexual couples are as good parents; Duty to have children;
       Child must care for ill parent; Main goal to make parents proud;
       Men make better political leaders; University more important for boys;
       Pre-school child suffers with working mother; Men make better business executives

[POLITICAL LEGITIMACY: AIMS]
Scale: 1=maintaining order; 2=more say in decisions; 3=fighting rising prices; 4=freedom of speech
Order: Aims first choice; Aims second choice

[POLITICAL ACTION]
Scale: 1=have done; 2=might do; 3=would never do
Order: Signing a petition; Joining boycotts; Attending demonstrations; Joining unofficial strikes

[INSTITUTIONAL CONFIDENCE]
Scale: 1=a lot; 2=quite a lot; 3=not very much; 4=not at all
Order: Churches; The press; Labour unions; The police; Civil services;
       Political parties; Major companies; Environmental movement; Justice system/courts

[POLITICAL SYSTEM]
Satisfaction scale: 1=dissatisfied → 10=satisfied
Preference scale: 1=very good; 2=fairly good; 3=fairly bad; 4=very bad
Order: Satisfaction with political system; Strong leader; Experts make decisions; Democratic system

[DEMOCRACY PERCEPTIONS]
Scale: 1=not an essential characteristic → 10=essential characteristic
Importance scale: 1=not important → 10=absolutely important
Order: Governments tax rich and subsidize poor; Religious authorities interpret laws;
       People choose leaders in free elections; State aid for unemployment;
       Army takes over when government incompetent; Civil rights protect liberty;
       Women have same rights as men; State makes incomes equal; People obey rulers;
       Importance of democracy; How democratically governed today (1=not democratic → 10=very democratic)

[ELECTION QUALITY]
Scale: 1=very often; 2=fairly often; 3=not often; 4=not at all often
Order: Votes counted fairly; Opposition candidates prevented; TV favors governing party;
       Voters bribed; Journalists fair; Election officials fair; Rich buy elections;
       Voters threatened with violence

[RELIGION]
Attendance scale: 1=more than once a week; 2=once a week; 3=once a month;
                  4=special holy days; 5=other holy days; 6=once a year; 7=less often; 8=never
Identity: 1=religious; 2=not religious; 3=atheist
Beliefs scale: 0=no; 1=yes
God importance scale: 1=not important at all → 10=absolutely important
Order: Attendance; Religious identity; Believe in God; Believe in life after death;
       Believe in hell; Believe in heaven; Importance of God in life

[JUSTIFIABLE BEHAVIORS]
Scale: 1=never justifiable → 10=always justifiable
Order: Claiming government benefits not entitled to; Avoiding fare on public transport;
       Cheating on taxes; Accepting a bribe; Homosexuality; Prostitution; Abortion;
       Divorce; Euthanasia; Suicide; Casual sex; Death penalty; Political violence

[TRUST]
General trust: 0=cannot be too careful; 1=most people can be trusted
Group trust scale: 1=trust completely; 2=trust somewhat; 3=not very much; 4=not at all
Order: General trust in people; Trust in family; Trust in neighbourhood;
       Trust in people you know; Trust in strangers; Trust in people of another religion;
       Trust in people of another nationality

[IMMIGRATION]
Scale: 1=very bad; 2=quite bad; 3=neither; 4=quite good; 5=very good
Order: Impact of immigrants on country development

[DEMOGRAPHICS]
Order: Sex (1=male; 2=female); Birth year; Age;
       Marital status (1=married; 2=living together; 3=divorced; 4=separated; 5=widowed; 6=single);
       Number of children; Household size;
       Education ISCED (0=less than primary → 8=doctoral);
       Education recoded (1=lower; 2=middle; 3=upper);
       Employment (0,NA=no paid work; 1=full time; 2=part time; 3=self employed;
                   4=retired; 5=housewife; 6=student; 7=unemployed; 8=other);
       Occupation sector (1=public; 2=private; 3=non-profit)

# TASK
Classify whether the respondent is from Eastern or Western Europe based only on the
provided values, beliefs, and attitudes.

# OUTPUT FORMAT
Respond with exactly this structure and nothing else:
Classification: East or West
Primary domain: [choose one-two and mention them through ";" in one column: religion | family | political legitimacy | trust | authority and autonomy | national identity]
Confidence: [high | medium | low]
Reason: [max 15 words, comma-separated facts only, no adjectives like "very" or "much", no comparisons]
"""

In [ ]:
# User prompt builder (converts one respondent row into LLM text)
def fmt(row, col):
    v = row.get(col, None) #safe function in case part of structured dataset breaks, to impute NA or change value format
    if pd.isna(v):
        return 'NA'
    if isinstance(v, float) and v.is_integer():
        return str(int(v))
    return str(v)

def fmt_group(row, col): #handles already-grouped value from the row
    v = row.get(col, None)
    if pd.isna(v):
        return 'NA'
    return str(v)

def row_to_user_prompt(row): #builds full user message for one respondent
    political_system = ';'.join([
        fmt(row, 'political_system_satisfaction'),
        fmt_group(row, 'political_system_pref')
    ])
    democracy = ';'.join([
        fmt_group(row, 'democracy_features'),
        fmt(row, 'democracy_importance'),
        fmt(row, 'democracy_current')
    ])
    religion = ';'.join([
        fmt(row, 'religious_attendance'),
        fmt(row, 'religious_identity'),
        fmt_group(row, 'religious_beliefs'),
        fmt(row, 'god_importance')
    ])
    trust = ';'.join([
        fmt(row, 'general_trust'),
        fmt_group(row, 'trust_groups')
    ])
    demographics = ';'.join([
        fmt(row, 'sex'),
        fmt(row, 'birth_year'),
        fmt(row, 'age'),
        fmt(row, 'marital_status'),
        fmt(row, 'n_children'),
        fmt(row, 'household_size'),
        fmt(row, 'education_isced'),
        fmt(row, 'education_r'),
        fmt(row, 'employment'),
        fmt(row, 'occupation_sector')
    ])
#assemble final prompt text, each section on its own line
    return (
        f"[IMPORTANCE IN LIFE] {fmt_group(row, 'importance_life')}\n"
        f"[CHILD QUALITIES] {fmt_group(row, 'child_qualities')}\n"
        f"[TOLERANCE] {fmt_group(row, 'tolerance')}\n"
        f"[FAMILY AND GENDER] {fmt_group(row, 'family_gender')}\n"
        f"[POLITICAL LEGITIMACY AIMS] {fmt_group(row, 'aims')}\n"
        f"[POLITICAL ACTION] {fmt_group(row, 'political_action')}\n"
        f"[INSTITUTIONAL CONFIDENCE] {fmt_group(row, 'confidence')}\n"
        f"[POLITICAL SYSTEM] {political_system}\n"
        f"[DEMOCRACY PERCEPTIONS] {democracy}\n"
        f"[ELECTION QUALITY] {fmt_group(row, 'election_quality')}\n"
        f"[RELIGION] {religion}\n"
        f"[JUSTIFIABLE BEHAVIORS] {fmt_group(row, 'justifiable')}\n"
        f"[TRUST] {trust}\n"
        f"[IMMIGRATION] {fmt(row, 'immigrant_impact')}\n"
        f"[DEMOGRAPHICS] {demographics}\n"
    )

## 4.b Prepare JSONL file

Each line is one respondent request. `custom_id` stores the uniqid so we can match results back to respondents.

In [ ]:
#this chunk is to split the whole file into n of chunk since limit for gpt5.1 is 900000 TPD
#Reset:delete tracking file to start from scratch ──
#if os.path.exists(path('submitted_batch_ids.json')):
#    os.remove(path('submitted_batch_ids.json'))
#    print("Tracking file deleted, starting from scratch")
#Safe chunk size under 900k token limit
BATCH_SIZE = 400  #400 ×2000 tokens approx. = 800k per chunk
MAX_BATCHES = 44 #limit set
BATCH_IDS_PATH = path('submitted_batch_ids.json')

# Load or initialize tracking
try:
    with open(BATCH_IDS_PATH) as f:
        submitted = json.load(f)
except:
    submitted = []

# Find where we left off
already_done = len(submitted) * BATCH_SIZE
remaining    = structured_dataset.iloc[already_done:].reset_index(drop=True)
batches_left = MAX_BATCHES - len(submitted)
n_batches    = (len(remaining) + BATCH_SIZE - 1) // BATCH_SIZE

print(f'Already submitted: {len(submitted)} batches ({already_done} respondents)')
print(f'Remaining:         {len(remaining)} respondents in {n_batches} batches')

for batch_idx in range(n_batches):
    start = batch_idx * BATCH_SIZE
    end   = min(start + BATCH_SIZE, len(remaining))
    chunk = remaining.iloc[start:end]

    # Build jsonl for this chunk
    jsonl_path = path(f'batch_requests_chunk_{len(submitted)+1}.jsonl')
    with open(jsonl_path, 'w') as f:
        for i, row in chunk.iterrows():
            request = {
                'custom_id': str(int(row['uniqid'])),
                'method': 'POST',
                'url': '/v1/chat/completions',
                'body': {
                    'model': 'gpt-5.1',
                    'max_completion_tokens': 200,
                    'temperature': 0,
                    'messages': [
                        {'role': 'system', 'content': SYSTEM_PROMPT},
                        {'role': 'user',   'content': row_to_user_prompt(row)}
                    ]
                }
            }
            f.write(json.dumps(request) + '\n')

    # Upload and submit
    with open(jsonl_path, 'rb') as f:
        batch_file = client_gpt.files.create(file=f, purpose='batch')

    batch_job = client_gpt.batches.create(
        input_file_id=batch_file.id,
        endpoint='/v1/chat/completions',
        completion_window='24h'
    )

    submitted.append({
        'chunk':     len(submitted) + 1,
        'batch_id':  batch_job.id,
        'start_row': already_done + start,
        'end_row':   already_done + end,
        'status':    'submitted'
    })

    # Save progress after every submission
    with open(BATCH_IDS_PATH, 'w') as f:
        json.dump(submitted, f, indent=2)

    print(f'Chunk {len(submitted)}/{n_batches + len(submitted) - batch_idx - 1} submitted: '
          f'{batch_job.id} ({len(chunk)} respondents)')

    # Wait for this batch to complete before submitting next
    # to stay within token limit
    print(f"Waiting for batch to complete...")
    while True:
        status = client_gpt.batches.retrieve(batch_job.id)
        print(f"  [{time.strftime("%H:%M:%S")}] {status.status} | "
                f"{status.request_counts.completed}/{status.request_counts.total}")
        if status.status in ["completed", "failed", "cancelled", "expired"]:
                submitted[-1]["status"] = status.status
                submitted[-1]["output_file_id"] = status.output_file_id
                with open(BATCH_IDS_PATH, "w") as f:
                    json.dump(submitted, f, indent=2)

                if status.status == "completed" and status.output_file_id:
                  os.makedirs(path('batches'), exist_ok=True)
                  content = client_gpt.files.content(status.output_file_id).text
                  out_path = path(f'batches/batch_output_chunk_{submitted[-1]["chunk"]}.jsonl')
                  with open(out_path, 'w') as f:
                      f.write(content)
                  print(f"  Saved output to {out_path}")
                break
        time.sleep(120) #two min to check update every time

    print(f'Chunk {len(submitted)} done with status: {status.status}\n')

print('All chunks submitted and completed!')
print(f'Batch IDs saved to: {BATCH_IDS_PATH}')

In [ ]:
#Load all saved batch outputs from our local folder

import pandas as pd
import json
import os

BASE_DIR = '/content/drive/MyDrive/thesis/'
BATCHES_DIR = BASE_DIR + 'batches/'   #folder where we saved batch output files
def path(f): return BASE_DIR + f

# Load the full dataset so we can rebuild the label lookup
structured_dataset = pd.read_csv(path('structured_dataset_stratified.csv'))
label_lookup = dict(zip(
    structured_dataset['uniqid'].astype(str),
    structured_dataset['east_west']
))

all_results = []

# Iterate over every .jsonl file saved in batches/
batch_files = sorted([f for f in os.listdir(BATCHES_DIR) if f.endswith('.jsonl')])
print(f"Found {len(batch_files)} batch output files in {BATCHES_DIR}")

for fname in batch_files:
    fpath = BATCHES_DIR + fname
    chunk_parsed = 0
    chunk_errors = 0

    with open(fpath, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                item = json.loads(line)
                uid    = item['custom_id']
                actual = 'East' if label_lookup.get(uid, 0) == 1 else 'West'

                # API-level error
                if item.get('error'):
                    all_results.append({
                        'uniqid': uid, 'source_file': fname,
                        'actual': actual, 'predicted': 'ERROR',
                        'correct': False, 'domain': 'NA',
                        'confidence': 'NA', 'reasoning': str(item['error'])
                    })
                    chunk_errors += 1
                    continue

                prediction_raw = item['response']['body']['choices'][0]['message']['content'].strip()

                classification, domain, confidence, reason = 'NA', 'NA', 'NA', 'NA'
                for line_r in prediction_raw.splitlines():
                    line_r = line_r.strip()
                    if line_r.startswith('Classification:'):
                        classification = line_r.split(':', 1)[1].strip()
                    elif line_r.startswith('Primary domain:'):
                        domain = line_r.split(':', 1)[1].strip()
                    elif line_r.startswith('Confidence:'):
                        confidence = line_r.split(':', 1)[1].strip()
                    elif line_r.startswith('Reason:'):
                        reason = line_r.split(':', 1)[1].strip()

                all_results.append({
                    'uniqid':     uid,
                    'source_file': fname,
                    'actual':     actual,
                    'predicted':  classification,
                    'correct':    classification == actual,
                    'domain':     domain,
                    'confidence': confidence,
                    'reasoning':  reason
                })
                chunk_parsed += 1

            except Exception as e:
                chunk_errors += 1
                all_results.append({
                    'uniqid': 'parse_error', 'source_file': fname,
                    'actual': 'NA', 'predicted': 'ERROR',
                    'correct': False, 'domain': 'NA',
                    'confidence': 'NA', 'reasoning': str(e)
                })

    print(f"  {fname}: parsed {chunk_parsed} | errors {chunk_errors}")

# Save merged CSV
results_df = pd.DataFrame(all_results)
results_df.to_csv(path('llm_results_all_selected_gpt-5.1.csv'), index=False)
print(f"\nTotal results: {len(results_df)}")
print(f"Saved to: {path('llm_results_all_selected_gpt-5.1.csv')}")

## 4.c Evaluation metrics

In [ ]:
# Load results
results_df       = pd.read_csv(path('llm_results_all_selected_gpt-5.1.csv'))
results_df_clean = results_df[results_df['predicted'].isin(['East', 'West'])].copy()

actual    = results_df_clean['actual']
predicted = results_df_clean['predicted']

print('Classification Report — Full Dataset (gpt-5.1 Batch API)')
print('=' * 55)
print(classification_report(actual, predicted, target_names=['East', 'West']))

# Save classification report as csv
report    = classification_report(actual, predicted, target_names=['East', 'West'], output_dict=True)
report_df = pd.DataFrame(report).transpose()
report_df.to_csv(path('classification_report_gpt-5.1_all.csv'))
print('Classification report saved.')

# Confusion matrix
cm   = confusion_matrix(actual, predicted, labels=['East', 'West'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['East', 'West'])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion Matrix — LLM East/West Classification (n={len(results_df_clean)})')
plt.tight_layout()
plt.savefig(path('confusion_matrix_all_gpt-5.1.png'), dpi=150)
plt.show()

print('\nDomain breakdown:')
print(results_df_clean['domain'].value_counts().to_string())

print('\nConfidence breakdown:')
print(results_df_clean['confidence'].value_counts().to_string())

print(f'\nTotal rows:    {len(results_df)}')
print(f'Valid rows:    {len(results_df_clean)}')
print(f'Invalid/Error: {len(results_df) - len(results_df_clean)}')
print(f'Accuracy:      {results_df_clean["correct"].mean() * 100:.1f}%')

In [ ]:
# Outcome distribution visualization (to check if all batches were loaded correctly)
valid   = len(results_df_clean)
invalid = len(results_df[results_df['predicted'] == 'NA'])
error   = len(results_df[results_df['predicted'] == 'ERROR'])
total   = valid + invalid + error

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart per outcome
categories = ['Valid (East/West)', 'Invalid (NA)', 'Error']
counts     = [valid, invalid, error]
colors     = ['steelblue', 'orange', 'red']

axes[0].bar(categories, counts, color=colors)
axes[0].set_ylabel('Respondents')
axes[0].set_title('Response outcome counts')
for idx, v in enumerate(counts):
    axes[0].text(idx, v + 5, str(v), ha='center', fontweight='bold')

# Pie chart global distribution
axes[1].pie(
    [v for v in counts if v > 0],
    labels=[c for c, v in zip(categories, counts) if v > 0],
    colors=[c for c, v in zip(colors, counts) if v > 0],
    autopct='%1.1f%%',
    startangle=90
)
axes[1].set_title('Global outcome distribution')

plt.tight_layout()
plt.savefig(path('batch_outcomes_gpt-5.1.png'), dpi=150)
plt.show()

print(f'Valid:   {valid} ({valid/total*100:.1f}%)')
print(f'Invalid: {invalid} ({invalid/total*100:.1f}%)')
print(f'Error:   {error} ({error/total*100:.1f}%)')

## 5.a Non-religious prompt

Now let's see how the accuracy will change if religion is to be ignored in the prompt

In [ ]:
# SYSTEM PROMPT
SYSTEM_PROMPT = """
# ROLE
You are analyzing a European Values Survey respondent. You will be given respondent
values in a fixed order. Each value corresponds to the matching attribute listed below.

# IMPORTANT RULES
- Classify based only on values, beliefs, and attitudes.
- Missing values are encoded as NA. Ignore them when reasoning.
- Values appear in exactly the same order as the listed attributes, separated by ";".
- The [RELIGION] section is provided but must be completely ignored. Do not use it in your reasoning or classification under any circumstances.
- Do not infer nationality from any single attribute — reason holistically.
- Reason must be under 15 words, facts only, no filler words, no comparisons to East or West.

# ATTRIBUTE DEFINITIONS

[IMPORTANCE IN LIFE]
Scale: 1=very important; 2=rather important; 3=not very important; 4=not important at all
Order: Family; Friends; Leisure time; Politics; Work; Religion

[CHILD QUALITIES]
Scale: 1=mentioned as important; 0=not mentioned
Order: Good manners; Independence; Hard work; Feeling of responsibility; Imagination;
       Tolerance and respect; Thrift; Determination; Religious faith; Unselfishness; Obedience

[TOLERANCE — neighbours]
Scale: 0=would not accept; 1=would accept
Order: People of a different race; Heavy drinkers; Immigrants; Drug addicts; Homosexuals

[FAMILY AND GENDER]
Scale: 1=strongly agree; 2=agree; 3=neither; 4=disagree; 5=strongly disagree
Order: Homosexual couples are as good parents; Duty to have children;
       Child must care for ill parent; Main goal to make parents proud;
       Men make better political leaders; University more important for boys;
       Pre-school child suffers with working mother; Men make better business executives

[POLITICAL LEGITIMACY: AIMS]
Scale: 1=maintaining order; 2=more say in decisions; 3=fighting rising prices; 4=freedom of speech
Order: Aims first choice; Aims second choice

[POLITICAL ACTION]
Scale: 1=have done; 2=might do; 3=would never do
Order: Signing a petition; Joining boycotts; Attending demonstrations; Joining unofficial strikes

[INSTITUTIONAL CONFIDENCE]
Scale: 1=a lot; 2=quite a lot; 3=not very much; 4=not at all
Order: Churches; The press; Labour unions; The police; Civil services;
       Political parties; Major companies; Environmental movement; Justice system/courts

[POLITICAL SYSTEM]
Satisfaction scale: 1=dissatisfied → 10=satisfied
Preference scale: 1=very good; 2=fairly good; 3=fairly bad; 4=very bad
Order: Satisfaction with political system; Strong leader; Experts make decisions; Democratic system

[DEMOCRACY PERCEPTIONS]
Scale: 1=not an essential characteristic → 10=essential characteristic
Importance scale: 1=not important → 10=absolutely important
Order: Governments tax rich and subsidize poor; Religious authorities interpret laws;
       People choose leaders in free elections; State aid for unemployment;
       Army takes over when government incompetent; Civil rights protect liberty;
       Women have same rights as men; State makes incomes equal; People obey rulers;
       Importance of democracy; How democratically governed today (1=not democratic → 10=very democratic)

[ELECTION QUALITY]
Scale: 1=very often; 2=fairly often; 3=not often; 4=not at all often
Order: Votes counted fairly; Opposition candidates prevented; TV favors governing party;
       Voters bribed; Journalists fair; Election officials fair; Rich buy elections;
       Voters threatened with violence

[RELIGION]
Attendance scale: 1=more than once a week; 2=once a week; 3=once a month;
                  4=special holy days; 5=other holy days; 6=once a year; 7=less often; 8=never
Identity: 1=religious; 2=not religious; 3=atheist
Beliefs scale: 0=no; 1=yes
God importance scale: 1=not important at all → 10=absolutely important
Order: Attendance; Religious identity; Believe in God; Believe in life after death;
       Believe in hell; Believe in heaven; Importance of God in life

[JUSTIFIABLE BEHAVIORS]
Scale: 1=never justifiable → 10=always justifiable
Order: Claiming government benefits not entitled to; Avoiding fare on public transport;
       Cheating on taxes; Accepting a bribe; Homosexuality; Prostitution; Abortion;
       Divorce; Euthanasia; Suicide; Casual sex; Death penalty; Political violence

[TRUST]
General trust: 0=cannot be too careful; 1=most people can be trusted
Group trust scale: 1=trust completely; 2=trust somewhat; 3=not very much; 4=not at all
Order: General trust in people; Trust in family; Trust in neighbourhood;
       Trust in people you know; Trust in strangers; Trust in people of another religion;
       Trust in people of another nationality

[IMMIGRATION]
Scale: 1=very bad; 2=quite bad; 3=neither; 4=quite good; 5=very good
Order: Impact of immigrants on country development

[DEMOGRAPHICS]
Order: Sex (1=male; 2=female); Birth year; Age;
       Marital status (1=married; 2=living together; 3=divorced; 4=separated; 5=widowed; 6=single);
       Number of children; Household size;
       Education ISCED (0=less than primary → 8=doctoral);
       Education recoded (1=lower; 2=middle; 3=upper);
       Employment (0,NA=no paid work; 1=full time; 2=part time; 3=self employed;
                   4=retired; 5=housewife; 6=student; 7=unemployed; 8=other);
       Occupation sector (1=public; 2=private; 3=non-profit)

# TASK
Classify whether the respondent is from Eastern or Western Europe based only on the
provided values, beliefs, and attitudes.

# OUTPUT FORMAT
Respond with exactly this structure and nothing else:
Classification: East or West
Primary domain: [choose one-two and mention them through ";" in one column: family | political legitimacy | trust | authority and autonomy | national identity]
Confidence: [high | medium | low]
Reason: [max 15 words, comma-separated facts only, no adjectives like "very" or "much", no comparisons]
"""

In [ ]:
# User prompt builder (converts one respondent row into LLM text)
def fmt(row, col):
    v = row.get(col, None) #safe function in case part of structured dataset breaks, to impute NA or change value format
    if pd.isna(v):
        return 'NA'
    if isinstance(v, float) and v.is_integer():
        return str(int(v))
    return str(v)

def fmt_group(row, col): #handles already-grouped value from the row
    v = row.get(col, None)
    if pd.isna(v):
        return 'NA'
    return str(v)

def row_to_user_prompt(row): #builds full user message for one respondent
    political_system = ';'.join([
        fmt(row, 'political_system_satisfaction'),
        fmt_group(row, 'political_system_pref')
    ])
    democracy = ';'.join([
        fmt_group(row, 'democracy_features'),
        fmt(row, 'democracy_importance'),
        fmt(row, 'democracy_current')
    ])
    religion = ';'.join([
        fmt(row, 'religious_attendance'),
        fmt(row, 'religious_identity'),
        fmt_group(row, 'religious_beliefs'),
        fmt(row, 'god_importance')
    ])
    trust = ';'.join([
        fmt(row, 'general_trust'),
        fmt_group(row, 'trust_groups')
    ])
    demographics = ';'.join([
        fmt(row, 'sex'),
        fmt(row, 'birth_year'),
        fmt(row, 'age'),
        fmt(row, 'marital_status'),
        fmt(row, 'n_children'),
        fmt(row, 'household_size'),
        fmt(row, 'education_isced'),
        fmt(row, 'education_r'),
        fmt(row, 'employment'),
        fmt(row, 'occupation_sector')
    ])
#assemble final prompt text, each section on its own line
    return (
        f"[IMPORTANCE IN LIFE] {fmt_group(row, 'importance_life')}\n"
        f"[CHILD QUALITIES] {fmt_group(row, 'child_qualities')}\n"
        f"[TOLERANCE] {fmt_group(row, 'tolerance')}\n"
        f"[FAMILY AND GENDER] {fmt_group(row, 'family_gender')}\n"
        f"[POLITICAL LEGITIMACY AIMS] {fmt_group(row, 'aims')}\n"
        f"[POLITICAL ACTION] {fmt_group(row, 'political_action')}\n"
        f"[INSTITUTIONAL CONFIDENCE] {fmt_group(row, 'confidence')}\n"
        f"[POLITICAL SYSTEM] {political_system}\n"
        f"[DEMOCRACY PERCEPTIONS] {democracy}\n"
        f"[ELECTION QUALITY] {fmt_group(row, 'election_quality')}\n"
        #RELIGION ABLATION:
        #f"[RELIGION] {religion}\n"
        f"[JUSTIFIABLE BEHAVIORS] {fmt_group(row, 'justifiable')}\n"
        f"[TRUST] {trust}\n"
        f"[IMMIGRATION] {fmt(row, 'immigrant_impact')}\n"
        f"[DEMOGRAPHICS] {demographics}\n"
    )

In [ ]:
import json
import time
import pandas as pd
#this chunk is to split the whole file into n of chunk since limit for gpt5.1 is 900000 TPD
# Safe chunk size under 900k token limit
path = lambda f: f"/content/drive/MyDrive/thesis/{f}"
structured_dataset = pd.read_csv(path('structured_dataset_stratified.csv'))
BATCH_SIZE = 400  #400 ×2000 tokens approx. = 800k per chunk
MAX_BATCHES = 44 #limit set
BATCH_IDS_PATH = path('submitted_batch_ids_no_religion.json')

# Load or initialize tracking
try:
    with open(BATCH_IDS_PATH) as f:
        submitted = json.load(f)
except:
    submitted = []

# Find where we left off
already_done = len(submitted) * BATCH_SIZE
remaining    = structured_dataset.iloc[already_done:].reset_index(drop=True)
batches_left = MAX_BATCHES - len(submitted)
n_batches    = (len(remaining) + BATCH_SIZE - 1) // BATCH_SIZE

print(f'Already submitted: {len(submitted)} batches ({already_done} respondents)')
print(f'Remaining:         {len(remaining)} respondents in {n_batches} batches')

for batch_idx in range(n_batches):
    start = batch_idx * BATCH_SIZE
    end   = min(start + BATCH_SIZE, len(remaining))
    chunk = remaining.iloc[start:end]

    # Build jsonl for this chunk
    jsonl_path = path(f'batches_no_religion/batch_requests_non_religious_chunk_{len(submitted)+1}.jsonl') #to save batches under the different name
    with open(jsonl_path, 'w') as f:
        for i, row in chunk.iterrows():
            request = {
                'custom_id': str(int(row['uniqid'])),
                'method': 'POST',
                'url': '/v1/chat/completions',
                'body': {
                    'model': 'gpt-5.1',
                    'max_completion_tokens': 200,
                    'temperature': 0,
                    'messages': [
                        {'role': 'system', 'content': SYSTEM_PROMPT},
                        {'role': 'user',   'content': row_to_user_prompt(row)}
                    ]
                }
            }
            f.write(json.dumps(request) + '\n')

    # Upload and submit
    with open(jsonl_path, 'rb') as f:
        batch_file = client_gpt.files.create(file=f, purpose='batch')

    batch_job = client_gpt.batches.create(
        input_file_id=batch_file.id,
        endpoint='/v1/chat/completions',
        completion_window='24h'
    )

    submitted.append({
        'chunk':     len(submitted) + 1,
        'batch_id':  batch_job.id,
        'start_row': already_done + start,
        'end_row':   already_done + end,
        'status':    'submitted'
    })

    # Save progress after every submission
    with open(BATCH_IDS_PATH, 'w') as f:
        json.dump(submitted, f, indent=2)

    print(f'Chunk {len(submitted)}/{n_batches + len(submitted) - batch_idx - 1} submitted: '
          f'{batch_job.id} ({len(chunk)} respondents)')

    # Wait for this batch to complete before submitting next
    # to stay within token limit
    print(f"Waiting for batch to complete...")
    while True:
        status = client_gpt.batches.retrieve(batch_job.id)
        print(f"  [{time.strftime("%H:%M:%S")}] {status.status} | "
                f"{status.request_counts.completed}/{status.request_counts.total}")
        if status.status in ["completed", "failed", "cancelled", "expired"]:
                submitted[-1]["status"] = status.status
                submitted[-1]["output_file_id"] = status.output_file_id
                with open(BATCH_IDS_PATH, "w") as f:
                    json.dump(submitted, f, indent=2)
                break
        time.sleep(120) #two min to check update every time

    print(f'Chunk {len(submitted)} done with status: {status.status}\n')

print('All chunks submitted and completed!')
print(f'Batch IDs saved to: {BATCH_IDS_PATH}')

In [ ]:
# Build lookup: uniqid: actual east_west label
label_lookup = dict(zip(
    structured_dataset['uniqid'].astype(str),
    structured_dataset['east_west']
))

# Load submitted batch IDs
with open(BATCH_IDS_PATH) as f:
    submitted = json.load(f)

all_results = []

for chunk in submitted:
    print(f'Processing chunk {chunk["chunk"]} — status: {chunk["status"]}')

    if chunk['status'] != 'completed' or not chunk.get('output_file_id'):
        print(f'  Skipped — not completed or no output file')
        continue

    # Download results for this chunk
    content = client_gpt.files.content(chunk['output_file_id']).text

    chunk_parsed = 0
    chunk_errors = 0

    for line in content.strip().splitlines():
        try:
            item = json.loads(line)
            uid  = item['custom_id']
            actual = 'East' if label_lookup.get(uid, 0) == 1 else 'West'

            # API level error
            if item.get('error'):
                all_results.append({
                    'uniqid': uid, 'chunk': chunk['chunk'],
                    'actual': actual, 'predicted': 'ERROR',
                    'correct': False, 'domain': 'NA',
                    'confidence': 'NA', 'reasoning': str(item['error'])
                })
                chunk_errors += 1
                continue

            prediction_raw = item['response']['body']['choices'][0]['message']['content'].strip()

            # Parse structured output
            classification, domain, confidence, reason = 'NA', 'NA', 'NA', 'NA'
            for line_r in prediction_raw.splitlines():
                line_r = line_r.strip()
                if line_r.startswith('Classification:'):
                    classification = line_r.split(':', 1)[1].strip()
                elif line_r.startswith('Primary domain:'):
                    domain = line_r.split(':', 1)[1].strip()
                elif line_r.startswith('Confidence:'):
                    confidence = line_r.split(':', 1)[1].strip()
                elif line_r.startswith('Reason:'):
                    reason = line_r.split(':', 1)[1].strip()

            all_results.append({
                'uniqid':     uid,
                'chunk':      chunk['chunk'],
                'actual':     actual,
                'predicted':  classification,
                'correct':    classification == actual,
                'domain':     domain,
                'confidence': confidence,
                'reasoning':  reason
            })
            chunk_parsed += 1

        except Exception as e:
            chunk_errors += 1
            all_results.append({
                'uniqid': 'parse_error', 'chunk': chunk['chunk'],
                'actual': 'NA', 'predicted': 'ERROR',
                'correct': False, 'domain': 'NA',
                'confidence': 'NA', 'reasoning': str(e)
            })

    print(f'  Parsed: {chunk_parsed} | Errors: {chunk_errors}')

# Save merged results
results_df = pd.DataFrame(all_results)
results_df.to_csv(path('llm_results_all_selected_gpt-5.1_no_religion.csv'), index=False)

print(f'\nTotal results: {len(results_df)}')
print(f'Saved to: {path("llm_results_all_selected_gpt-5.1_no_religion.csv")}')

In [ ]:
# Load results
results_df       = pd.read_csv(path('llm_results_all_selected_gpt-5.1_no_religion.csv'))
results_df_clean = results_df[results_df['predicted'].isin(['East', 'West'])].copy()

actual    = results_df_clean['actual']
predicted = results_df_clean['predicted']

print('Classification Report — Full Dataset (gpt-5.1 Batch API)')
print('=' * 55)
print(classification_report(actual, predicted, target_names=['East', 'West']))

# Save classification report as csv
report    = classification_report(actual, predicted, target_names=['East', 'West'], output_dict=True)
report_df = pd.DataFrame(report).transpose()
report_df.to_csv(path('classification_report_gpt-5.1_all_no_religion.csv'))
print('Classification report saved.')

# Confusion matrix
cm   = confusion_matrix(actual, predicted, labels=['East', 'West'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['East', 'West'])

fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Confusion Matrix — LLM East/West Classification (n={len(results_df_clean)})')
plt.tight_layout()
plt.savefig(path('confusion_matrix_all_gpt-5.1_no_religion.png'), dpi=150)
plt.show()

print('\nDomain breakdown:')
print(results_df_clean['domain'].value_counts().to_string())

print('\nConfidence breakdown:')
print(results_df_clean['confidence'].value_counts().to_string())

print(f'\nTotal rows:    {len(results_df)}')
print(f'Valid rows:    {len(results_df_clean)}')
print(f'Invalid/Error: {len(results_df) - len(results_df_clean)}')
print(f'Accuracy:      {results_df_clean["correct"].mean() * 100:.1f}%')

In [ ]:
# Outcome distribution visualization
valid   = len(results_df_clean)
invalid = len(results_df[results_df['predicted'] == 'NA'])
error   = len(results_df[results_df['predicted'] == 'ERROR'])
total   = valid + invalid + error

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart per outcome
categories = ['Valid (East/West)', 'Invalid (NA)', 'Error']
counts     = [valid, invalid, error]
colors     = ['steelblue', 'orange', 'red']

axes[0].bar(categories, counts, color=colors)
axes[0].set_ylabel('Respondents')
axes[0].set_title('Response outcome counts')
for idx, v in enumerate(counts):
    axes[0].text(idx, v + 5, str(v), ha='center', fontweight='bold')

# Pie chart global distribution
axes[1].pie(
    [v for v in counts if v > 0],
    labels=[c for c, v in zip(categories, counts) if v > 0],
    colors=[c for c, v in zip(colors, counts) if v > 0],
    autopct='%1.1f%%',
    startangle=90
)
axes[1].set_title('Global outcome distribution')

plt.tight_layout()
plt.savefig(path('batch_outcomes_gpt-5.1.png'), dpi=150)
plt.show()

print(f'Valid:   {valid} ({valid/total*100:.1f}%)')
print(f'Invalid: {invalid} ({invalid/total*100:.1f}%)')
print(f'Error:   {error} ({error/total*100:.1f}%)')

## 5.b Table domain comparison

In [ ]:
import pandas as pd
import numpy as np

path = lambda f: f"/content/drive/MyDrive/thesis/{f}"

# load both files
with_religion = pd.read_csv(path("llm_results_all_selected_gpt-5.1.csv"))
no_religion = pd.read_csv(path("llm_results_all_selected_gpt-5.1_no_religion.csv"))

def normalize_domain(domain):
    """Merge domains with same values but different order"""
    if pd.isna(domain) or domain == '':
        return None
    parts = str(domain).split(';')
    parts = [p.strip() for p in parts]
    parts.sort()
    return ';'.join(parts)

# process with_religion
with_religion['domain_normalized'] = with_religion['domain'].apply(normalize_domain)
domain_counts_with = with_religion['domain_normalized'].value_counts()

# process no_religion
no_religion['domain_normalized'] = no_religion['domain'].apply(normalize_domain)
domain_counts_no = no_religion['domain_normalized'].value_counts()

# create comparison dataframe
all_domains = set(domain_counts_with.index) | set(domain_counts_no.index)
all_domains = [d for d in all_domains if d]

comparison_df = pd.DataFrame({
    'Domain': all_domains,
    'With Religion': [domain_counts_with.get(d, 0) for d in all_domains],
    'No Religion': [domain_counts_no.get(d, 0) for d in all_domains],
})

comparison_df['Total'] = comparison_df['With Religion'] + comparison_df['No Religion']

# sort by Total in descending order
comparison_df = comparison_df.sort_values('Total', ascending=False).reset_index(drop=True)

# print table
print("="*100)
print("DOMAIN FREQUENCY COMPARISON (sorted by total frequency)")
print("="*100)
print(f"\n{'Domain':<50s} {'With Religion':>15s} {'No Religion':>15s} {'Total':>10s}")
print("-" * 100)

for _, row in comparison_df.iterrows():
    print(f"{row['Domain']:<50s} {row['With Religion']:>15d} {row['No Religion']:>15d} {row['Total']:>10d}")

print("-" * 100)
with_total = comparison_df['With Religion'].sum()
no_total = comparison_df['No Religion'].sum()
grand_total = comparison_df['Total'].sum()
print(f"{'TOTAL':<50s} {with_total:>15d} {no_total:>15d} {grand_total:>10d}")

# save to CSV
comparison_df.to_csv(path("domain_comparison_normalized.csv"), index=False)
print("\n\nComparison table saved to domain_comparison_normalized.csv")

## 6. SHAP proxy evaluation

Now we move on with objective two to investiagte which respondent characteristics are associated with the LLM making an error. For this we expand on the SHAP algorithm and merge LLM-error binary classifier with original clean dataset to inspect which specific variables correlte with this missclassification.

It is worth noting this approach does not explicitly explain the LLM's internal reasoning, but rather serves as a proxy. A surrogate XGBoost classifier is trained on the merged dataset to predict the binary error flag from raw survey variables, after which SHAP values are extracted to quantify each variable's marginal contribution to misclassification likelihood.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

path = lambda f: f"/content/drive/MyDrive/thesis/{f}"

#load data
llm = pd.read_csv(path("llm_results_all_selected_gpt-5.1.csv"))
clean = pd.read_csv(path("clean_dataset.csv"))

#align and prepare data
llm["uniqid"] = llm["uniqid"].astype(str).str.strip()
clean["uniqid"] = clean["uniqid"].astype(str).str.strip()

llm["error"] = (~llm["correct"].astype(bool)).astype(int)

merged = llm[["uniqid", "error"]].merge(clean, on="uniqid", how="inner")

drop_cols = ["uniqid", "year", "pwght", "reg_nuts2", "east_west"]
feature_cols = [c for c in clean.columns if c not in drop_cols + ["uniqid"]]

X = merged[feature_cols].copy()
y = merged["error"]

X = X.replace("NA", np.nan)
X = X.apply(pd.to_numeric, errors="coerce")

#train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# train XGBOOST
clf = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
    random_state=42,
    eval_metric="logloss",
    early_stopping_rounds=20,
    verbosity=0,
)

clf.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

print("\nsurrogate model performance:")
print(classification_report(y_test, clf.predict(X_test), target_names=["correct", "error"]))

#compute shap values
explainer = shap.TreeExplainer(clf)
shap_values = explainer(X_test)

print(f"\nshap values computed for {len(X_test):,} respondents")

#feature name mapping
feature_names_dict = {
    "F118": "Homosexuality justifiability",
    "X025A_01": "Education level",
    "E069_06": "Confidence in parties",
    "E114": "Preference for strong leader",
    "D081": "Same-sex couples as parents",
    "F063": "Importance of God",
    "D026_05": "Child care for parents",
    "F050": "Belief in God",
    "E069_01": "Confidence in churches",
    "E265_01": "Election fairness",
    "E236": "Democratic governance",
    "E228": "Equal incomes by state",
    "X052": "Occupation sector",
    "E224": "Tax rich to aid poor",
    "F121": "Justifiability of divorce",
    "E265_06": "Election officials fair",
    "A030": "Hard work as value",
    "G052": "Immigrant impact",
    "E069_13": "Confidence in companies",
    "F034": "Religious identity",
}

# Create readable names for all features
readable_names = [feature_names_dict.get(col, col) for col in X_test.columns]

#top 10 bar chart
fig, ax = plt.subplots(figsize=(10, 7))
shap.plots.bar(shap_values, max_display=10, ax=ax, show=False)

# Get current y-axis labels and replace only the feature ones
current_labels = [t.get_text() for t in ax.get_yticklabels()]
new_labels = []

for label in current_labels:
    if label in feature_names_dict:
        new_labels.append(feature_names_dict[label])
    else:
        new_labels.append(label)  # Keep "Sum of other features" as is

ax.set_yticklabels(new_labels)
ax.set_title(
    "Top 10 survey variables associated with LLM misclassification\n(mean |SHAP value| across test respondents)",
    fontsize=11
)
plt.tight_layout()
plt.savefig(path("shap_bar_top10.png"), dpi=150, bbox_inches="tight")
plt.show()
print("saved: shap_bar_top10.png")

#10 beeswarm
fig = plt.figure(figsize=(10, 7))
shap.plots.beeswarm(shap_values, max_display=10, show=False, plot_size=(10, 7))

ax = plt.gca()
current_labels = [t.get_text() for t in ax.get_yticklabels()]
new_labels = []

for label in current_labels:
    if label in feature_names_dict:
        new_labels.append(feature_names_dict[label])
    else:
        new_labels.append(label)

ax.set_yticklabels(new_labels)
plt.title(
    "SHAP beeswarm — direction and magnitude of each feature's effect on error probability",
    fontsize=11
)
plt.tight_layout()
plt.savefig(path("shap_beeswarm_top10.png"), dpi=150, bbox_inches="tight")
plt.show()
print("saved: shap_beeswarm_top10.png")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

path = lambda f: f"/content/drive/MyDrive/thesis/{f}"

#load data
llm = pd.read_csv(path("llm_results_all_selected_gpt-5.1_no_religion.csv"))
clean = pd.read_csv(path("clean_dataset.csv"))

#align and prepare data
llm["uniqid"] = llm["uniqid"].astype(str).str.strip()
clean["uniqid"] = clean["uniqid"].astype(str).str.strip()

llm["error"] = (~llm["correct"].astype(bool)).astype(int)

merged = llm[["uniqid", "error"]].merge(clean, on="uniqid", how="inner")

drop_cols = ["uniqid", "year", "pwght", "reg_nuts2", "east_west"]
feature_cols = [c for c in clean.columns if c not in drop_cols + ["uniqid"]]

X = merged[feature_cols].copy()
y = merged["error"]

X = X.replace("NA", np.nan)
X = X.apply(pd.to_numeric, errors="coerce")

#train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

#xgboost
clf = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
    random_state=42,
    eval_metric="logloss",
    early_stopping_rounds=20,
    verbosity=0,
)

clf.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

print("\nsurrogate model performance:")
print(classification_report(y_test, clf.predict(X_test), target_names=["correct", "error"]))

#compute shap values
explainer = shap.TreeExplainer(clf)
shap_values = explainer(X_test)

print(f"\nshap values computed for {len(X_test):,} respondents")

#feature map features
feature_names_dict = {
    "F118": "Homosexuality justifiability",
    "X025A_01": "Education level",
    "E069_06": "Confidence in parties",
    "E114": "Preference for strong leader",
    "D081": "Same-sex couples as parents",
    "F063": "Importance of God",
    "D026_05": "Child care for parents",
    "F050": "Belief in God",
    "E069_01": "Confidence in churches",
    "E265_01": "Election fairness",
    "E236": "Democratic governance",
    "E228": "Equal incomes by state",
    "X052": "Occupation sector",
    "E224": "Tax rich to aid poor",
    "F121": "Justifiability of divorce",
    "E265_06": "Election officials fair",
    "A030": "Hard work as value",
    "G052": "Immigrant impact",
    "E069_13": "Confidence in companies",
    "F034": "Religious identity",
}

# Create readable names for all features
readable_names = [feature_names_dict.get(col, col) for col in X_test.columns]

#top 10 bar chart
fig, ax = plt.subplots(figsize=(10, 7))
shap.plots.bar(shap_values, max_display=10, ax=ax, show=False)

# Get current y-axis labels and replace only the feature ones
current_labels = [t.get_text() for t in ax.get_yticklabels()]
new_labels = []

for label in current_labels:
    if label in feature_names_dict:
        new_labels.append(feature_names_dict[label])
    else:
        new_labels.append(label)  # Keep "Sum of other features" as is

ax.set_yticklabels(new_labels)
ax.set_title(
    "Top 10 survey variables associated with LLM misclassification\n(mean |SHAP value| across test respondents)",
    fontsize=11
)
plt.tight_layout()
plt.savefig(path("shap_bar_top10_no_religion.png"), dpi=150, bbox_inches="tight")
plt.show()
print("saved: shap_bar_top10_no_religion.png")

#10 beeswarm
fig = plt.figure(figsize=(10, 7))
shap.plots.beeswarm(shap_values, max_display=10, show=False, plot_size=(10, 7))

ax = plt.gca()
current_labels = [t.get_text() for t in ax.get_yticklabels()]
new_labels = []

for label in current_labels:
    if label in feature_names_dict:
        new_labels.append(feature_names_dict[label])
    else:
        new_labels.append(label)

ax.set_yticklabels(new_labels)
plt.title(
    "SHAP beeswarm — direction and magnitude of each feature's effect on error probability",
    fontsize=11
)
plt.tight_layout()
plt.savefig(path("shap_beeswarm_top10_no_religion.png"), dpi=150, bbox_inches="tight")
plt.show()
print("saved: shap_beeswarm_top10_no_religion.png")

Lastly, to recognise high-low centering of features across East-West, we visualize the values separately for both scenarios:  

In [ ]:
feature_names_dict = {
    "F118": "Homosexuality justifiability", "X025A_01": "Education level",
    "E069_06": "Confidence in parties", "E114": "Preference for strong leader",
    "D081": "Same-sex couples as parents", "F063": "Importance of God",
    "D026_05": "Child care for parents", "F050": "Belief in God",
    "E069_01": "Confidence in churches", "E265_01": "Election fairness",
    "E236": "Democratic governance", "E228": "Equal incomes by state",
    "X052": "Occupation sector", "E224": "Tax rich to aid poor",
    "F121": "Justifiability of divorce", "E265_06": "Election officials fair",
    "A030": "Hard work as value", "G052": "Immigrant impact",
    "E069_13": "Confidence in companies", "F034": "Religious identity",
}

# native SHAP colors: red for East-leaning bars, blue for West-leaning
EAST_COLOR, WEST_COLOR = "#ff0d57", "#008bfb"

# features to plot, in the exact order they should appear (top -> bottom)
FEATURES_WITH_RELIGION = [
    "F118", "X025A_01", "E069_06", "E114", "D081",
    "F063", "D026_05", "F050", "E069_01",
]
FEATURES_NO_RELIGION = [
    "F118", "X025A_01", "E069_06", "G052", "D081",
    "E265_01", "A030", "X052", "E236",
]


def region_difference_plot(llm_file, scenario_label, features, east_value="East"):
    llm = pd.read_csv(path(llm_file))
    clean = pd.read_csv(path("clean_dataset.csv"))
    llm["uniqid"] = llm["uniqid"].astype(str).str.strip()
    clean["uniqid"] = clean["uniqid"].astype(str).str.strip()
    llm["error"] = (~llm["correct"].astype(bool)).astype(int)

    merged = llm[["uniqid", "error", "actual"]].merge(clean, on="uniqid", how="inner")

    drop_cols = ["uniqid", "year", "pwght", "reg_nuts2", "east_west"]
    feature_cols = [c for c in clean.columns if c not in drop_cols + ["uniqid"]]
    X = merged[feature_cols].replace("NA", np.nan).apply(pd.to_numeric, errors="coerce")
    y = merged["error"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    ew_test = merged.loc[X_test.index, "actual"]
    print(f"\n[{scenario_label}] region counts:\n{ew_test.value_counts(dropna=False)}")
    mask_east = (ew_test == east_value).values
    if mask_east.sum() == 0:
        raise ValueError(f"No rows matched east_value={east_value!r}; check counts above.")

    clf = XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
        random_state=42, eval_metric="logloss",
        early_stopping_rounds=20, verbosity=0,
    )
    clf.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

    shap_values = shap.TreeExplainer(clf)(X_test)
    cols = X_test.columns.tolist()
    col_idx = {c: i for i, c in enumerate(cols)}

    missing = [f for f in features if f not in col_idx]
    if missing:
        raise ValueError(f"These features are not in the data: {missing}")

    imp_e = np.abs(shap_values[mask_east].values).mean(0)
    imp_w = np.abs(shap_values[~mask_east].values).mean(0)
    diff = imp_e - imp_w                        # >0 => drives error more for East

    idx = [col_idx[f] for f in features]        # keep the requested order
    feats = [feature_names_dict.get(f, f) for f in features]
    vals = diff[idx]
    colors = [EAST_COLOR if v > 0 else WEST_COLOR for v in vals]

    n = len(features)
    ypos = np.arange(n)
    fig, ax = plt.subplots(figsize=(9, 0.5 * n + 1.6))
    ax.barh(ypos, vals, color=colors, height=0.65, zorder=3)

    # value labels at the end of each bar, in the bar's colour (SHAP style)
    span = max(abs(vals.min()), abs(vals.max()), 1e-9)
    pad = span * 0.02
    for yp, v, c in zip(ypos, vals, colors):
        ax.text(v + pad if v >= 0 else v - pad, yp, f"{v:+.2f}",
                va="center", ha="left" if v >= 0 else "right",
                color=c, fontsize=9, zorder=4)

    ax.set_yticks(ypos)
    ax.set_yticklabels(feats)
    ax.invert_yaxis()                           # first feature at the top
    ax.tick_params(axis="y", length=0)          # no y tick marks
    ax.axvline(0, color="#999999", lw=0.8, zorder=2)

    # minimal frame: keep only the bottom axis, like the SHAP bar plot
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.set_xlim(-span * 1.25, span * 1.25)      # room for the labels

    ax.set_xlabel("mean(|SHAP| difference)   East − West")
    plt.tight_layout()
    plt.savefig(path(f"shap_diff_{scenario_label}.png"), dpi=150, bbox_inches="tight")
    plt.show()


region_difference_plot("llm_results_all_selected_gpt-5.1.csv",
                       "with_religion", FEATURES_WITH_RELIGION)
region_difference_plot("llm_results_all_selected_gpt-5.1_no_religion.csv",
                       "no_religion", FEATURES_NO_RELIGION)

## 7. Spatial analysis

Having performed the full LLM error evaluation and SHAP feautre selection contributing the most to those errors, it is time to check whether the failed outputs follow spatial dependeny. This way we will track if particular regions are skewed during prompting and in which areas those errors cluster particulary.

In [ ]:
#to the output ids, we will first map the countries and regions back
import pandas as pd

#load both files
llm = pd.read_csv("/content/drive/MyDrive/thesis/llm_results_all_selected_gpt-5.1.csv")
geo = pd.read_csv("/content/drive/MyDrive/thesis/east_west_dataset.csv")

print(f"LLM results:  {llm.shape[0]:,} rows x {llm.shape[1]} cols")
print(f"Geo dataset:  {geo.shape[0]:,} rows x {geo.shape[1]} cols")

#cast uniqid to string in both files before merging to avoid type mismatch
llm["uniqid"] = llm["uniqid"].astype(str)
geo["uniqid"] = geo["uniqid"].astype(str)

#only need these three columns from geo, drop duplicates just in case
geo_slim = geo[["uniqid", "cntry", "reg_nuts2", "gwght"]].drop_duplicates(subset="uniqid")

#left join so i dont lose any llm rows even if geo match is missing
merged = llm.merge(geo_slim, on="uniqid", how="left")

#quick check how many rows got matched vs not
n_matched   = merged["cntry"].notna().sum()
n_unmatched = merged["cntry"].isna().sum()
print(f"\nmatched: {n_matched:,}  |  unmatched: {n_unmatched:,}")

#if something didnt match, print a few ids to see whats wrong
if n_unmatched > 0:
    print("unmatched uniqids sample:")
    print(merged.loc[merged["cntry"].isna(), "uniqid"].head(10).tolist())

#drop unmatched rows where cntry is missing
merged = merged.dropna(subset=["cntry"])

#recalculate totals after drop
print(f"rows after dropping unmatched: {merged.shape[0]:,}")
print(f"rows dropped: {n_unmatched:,}")
print(f"remaining cols: {merged.shape[1]}")

#save
merged.to_csv("/content/drive/MyDrive/thesis/spatial_clean.csv", index=False)
print(f"\ndone: spatial_clean.csv  |  {merged.shape[0]:,} rows x {merged.shape[1]} cols")

In [ ]:
#some reg_nuts2 values are missing, let's identify which countries had missing values in region variable first and how many:

df = pd.read_csv("/content/drive/MyDrive/thesis/spatial_clean.csv")

#find all rows where reg_nuts2 is not a proper code
missing = df[~df['reg_nuts2'].str.match(r'^[A-Z]', na=False)]

#summarize by country and missing code value
summary = missing.groupby(['cntry', 'reg_nuts2']).size().reset_index(name='count')
print(summary.to_string(index=False))
print(f"\ntotal missing: {len(missing)}")

In [ ]:
#now, let's imput missing regions with nuts2 code of capitals of those countries:
#mapping based on country number -> nuts2 capital region
#fill in the right codes from your codebook before running
capital_map = {
    276: "DE30",   #germany -- Berlin
    826: "UKI3",   #uk -- inner London
    804: "UA3",   #ukraine -- Kyiv
    51:  "AM01",   #armenia -- Yerevan
    528: "NL326",   #netherlands -- Amsterdam
    352: "IS00",   #iceland -- Reykjavik
}

#only touch rows that are actually missing
mask = ~df['reg_nuts2'].str.match(r'^[A-Z]', na=False)

df.loc[mask, 'reg_nuts2'] = df.loc[mask, 'cntry'].map(capital_map)

#verify nothing is left unimputed
still_missing = df[~df['reg_nuts2'].str.match(r'^[A-Z]', na=False)]
print(f"still missing after imputation: {len(still_missing)}")

df.to_csv("/content/drive/MyDrive/thesis/spatial_clean.csv", index=False)
print("saved: spatial_clean.csv")

Next, we download nuts2 2024 shapefile directly from eurostat gisco to match regions to their centroids, then identify non-EUregions where manual centroid computation is needed. Before computing centroids, it is also of crucial importance to project all centroids to standard EU projection EPSG:3035 to avoid wrong CRS warnings from Python. Then we flip back to WGS84 (4326) so the coordinates are regular lat/lon that everything else can work with.

In [ ]:
import pandas as pd
import geopandas as gpd
import requests

#download nuts2 2024 shapefile directly from eurostat gisco
url = "https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/NUTS_RG_20M_2024_4326_LEVL_2.geojson"
response = requests.get(url)
with open("/content/drive/MyDrive/thesis/nuts2_2024.geojson", "wb") as f:
    f.write(response.content)
print("shapefile downloaded")

#load shapefile
nuts2 = gpd.read_file("/content/drive/MyDrive/thesis/nuts2_2024.geojson")

#project to epsg:3035 (eu standard metric projection) before computing centroids
#then convert centroid coords back to wgs84 lat/lon for usability
nuts2_proj = nuts2.to_crs(epsg=3035)
nuts2_proj["centroid"] = nuts2_proj.geometry.centroid
centroids_wgs84 = nuts2_proj["centroid"].to_crs(epsg=4326)
nuts2["lon"] = centroids_wgs84.x
nuts2["lat"] = centroids_wgs84.y

nuts2_slim = nuts2[["NUTS_ID", "lon", "lat"]].copy()

#load our data
df = pd.read_csv("/content/drive/MyDrive/thesis/spatial_clean.csv")

#drop lon/lat if already there so re-running doesnt duplicate or conflict
for col in ["lon", "lat", "NUTS_ID"]:
    if col in df.columns:
        df = df.drop(columns=col)

#join centroids by reg_nuts2
df = df.merge(nuts2_slim, left_on="reg_nuts2", right_on="NUTS_ID", how="left")
df = df.drop(columns="NUTS_ID")

#check how many regions didnt match the shapefile
no_match = df[df["lon"].isna()]["reg_nuts2"].unique()
print(f"regions with no geometry match ({len(no_match)}): {no_match}")

#save
df.to_csv("/content/drive/MyDrive/thesis/spatial_clean.csv", index=False)
print(f"done -- spatial_clean.csv | {df.shape[0]:,} rows x {df.shape[1]} cols")

Based on the output, we see the three categories of mismatches: UK codes (UKI5, UKM7 etc.), which exists due to Brexit, UK left Eurostat NUTS after 2021, so they're gone from the 2024 shapefile. Next, non-EU countries (GE, AM, AZ, RU, BY, UA, BA). Those never were in NUTS2 and require manual imputation. Lastly,
old codes (UA63, UA61, UA31 etc.): the 2024 recode that is to happen since january 2027, so the shapefile doesn't pick it up already (changes mentioned here: https://ec.europa.eu/eurostat/web/nuts)

In [ ]:
#first let's solve the problem with old recoding of nuts2 codes to 2024 versions and see how many more regions are left to handle:
#norway mergers and croatia recode based on eurostat 2024 changes
#reload fresh to avoid stale state from previous runs
df = pd.read_csv("/content/drive/MyDrive/thesis/spatial_clean.csv")

#drop rows where reg_nuts2 is null before doing anything
df = df.dropna(subset=["reg_nuts2"])
print(f"rows after dropping null reg_nuts2: {df.shape[0]:,}")

#recode old nuts2 codes to 2024 versions
recode_map = {
    #norway
    "NO01": "NO0D",
    "NO03": "NO0B",
    "NO04": "NO0A",
    "NO05": "NO0C",
    #croatia
    "HR04": "HR05",
}

df["reg_nuts2"] = df["reg_nuts2"].replace(recode_map)

#verify those codes are gone
still_old = df[df["reg_nuts2"].isin(recode_map.keys())]["reg_nuts2"].unique()
print(f"old codes still present: {still_old if len(still_old) > 0 else 'none, all recoded'}")

#reload shapefile to check how many regions are still unmatched after recode
nuts2 = gpd.read_file("/content/drive/MyDrive/thesis/nuts2_2024.geojson")
valid_codes = set(nuts2["NUTS_ID"])

unmatched = df[~df["reg_nuts2"].isin(valid_codes)]["reg_nuts2"].unique()
unmatched_rows = df[~df["reg_nuts2"].isin(valid_codes)].shape[0]
print(f"\nregions still unmatched: {len(unmatched)} unique codes | {unmatched_rows} rows")

#sorted only on non-null string values
print(f"regions left to manually handle: {sorted([x for x in unmatched if isinstance(x, str)])}")

df.to_csv("/content/drive/MyDrive/thesis/spatial_clean.csv", index=False)
print("\nsaved: spatial_clean.csv")

In [ ]:
#now, let's check 2021 shapefile that previously covered Great Britain, Netherlands and Portugal regions to see if we have matches to save them to our file:
import requests, geopandas as gpd, io

#try 2021 vintage for leftover eu/candidate codes
url_2021 = "https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/NUTS_RG_20M_2021_4326_LEVL_2.geojson"
r = requests.get(url_2021)
nuts2021 = gpd.read_file(io.BytesIO(r.content))

still_missing = ['AM01','AM02','AM03','AM04','AZ-ABS','AZ-AGS','AZ-AGU','AZ-AST','AZ-BA','AZ-BAL','AZ-BAR','AZ-BEY','AZ-BIL','AZ-GA','AZ-GAD','AZ-GOR','AZ-GOY','AZ-GYG','AZ-HAC','AZ-IMI','AZ-KUR','AZ-LAN','AZ-MAS','AZ-MI','AZ-NEF','AZ-QAB','AZ-QAZ','AZ-QUS','AZ-SAB','AZ-SAL','AZ-SAT','AZ-SKR','AZ-SM','AZ-SMX','AZ-TOV','AZ-XAC','AZ-ZAR','BABIH','BABRC','BASRP','BY01','BY02','BY03','BY04','BY05','BY06','BY07','GE01','GE02','GE03','NL31','NL326','NL33','NO0C','NO0D','PT16','PT17','PT18','RU11','RU21','RU31','RU41','RU51','RU61','RU71','RU81','UA11','UA12','UA13','UA14','UA21','UA23','UA24','UA3','UA31','UA41','UA42','UA51','UA52','UA53','UA61','UA62','UA63','UA71','UA72','UA73','UA81','UA82','UA83','UKC2','UKD3','UKD4','UKD7','UKE1','UKE3','UKE4','UKF1','UKF2','UKG1','UKG3','UKH1','UKH2','UKH3','UKI3','UKI5','UKI6','UKI7','UKJ1','UKJ2','UKJ4','UKK1','UKK4','UKL1','UKL2','UKM5','UKM7','UKM8']

valid_2021 = set(nuts2021["NUTS_ID"])
now_matched = [c for c in still_missing if c in valid_2021]
still_unmatched = [c for c in still_missing if c not in valid_2021]

print(f"picked up by 2021 shapefile: {len(now_matched)} -- {now_matched}")
print(f"\nstill unmatched: {len(still_unmatched)} -- {still_unmatched}")

In [ ]:
import pandas as pd
import geopandas as gpd
import requests, io

df = pd.read_csv("/content/drive/MyDrive/thesis/spatial_clean.csv")

#drop existing lon/lat to avoid conflicts on rerun
for col in ["lon", "lat", "NUTS_ID"]:
    if col in df.columns:
        df = df.drop(columns=col)

#load 2024 shapefile for main eu coverage
url_2024 = "https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/NUTS_RG_20M_2024_4326_LEVL_2.geojson"
nuts2024 = gpd.read_file(io.BytesIO(requests.get(url_2024).content))
nuts2024_proj = nuts2024.to_crs(epsg=3035)
nuts2024["lon"] = nuts2024_proj.geometry.centroid.to_crs(epsg=4326).x
nuts2024["lat"] = nuts2024_proj.geometry.centroid.to_crs(epsg=4326).y
nuts2024_slim = nuts2024[["NUTS_ID", "lon", "lat"]].copy()

#load 2021 shapefile for nl, pt, uk codes missing from 2024
url_2021 = "https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/NUTS_RG_20M_2021_4326_LEVL_2.geojson"
nuts2021 = gpd.read_file(io.BytesIO(requests.get(url_2021).content))
nuts2021_proj = nuts2021.to_crs(epsg=3035)
nuts2021["lon"] = nuts2021_proj.geometry.centroid.to_crs(epsg=4326).x
nuts2021["lat"] = nuts2021_proj.geometry.centroid.to_crs(epsg=4326).y
nuts2021_slim = nuts2021[["NUTS_ID", "lon", "lat"]].copy()

#combine both shapefiles, 2024 takes priority where codes overlap
combined = pd.concat([nuts2024_slim, nuts2021_slim]).drop_duplicates(subset="NUTS_ID", keep="first")
print(f"combined shapefile: {len(combined)} unique nuts2 codes")

#join to our data
df = df.merge(combined, left_on="reg_nuts2", right_on="NUTS_ID", how="left")
df = df.drop(columns="NUTS_ID")

#check whats still missing
still_missing = df[df["lon"].isna()]["reg_nuts2"].unique()
still_missing_rows = df[df["lon"].isna()].shape[0]
print(f"\nstill unmatched: {len(still_missing)} unique codes | {still_missing_rows} rows")
print(sorted(still_missing))

df.to_csv("/content/drive/MyDrive/thesis/spatial_clean.csv", index=False)
print("\nsaved: spatial_clean.csv")

In [ ]:
#Since eurostat shap file does not contain all the information about the Eastern-European countries, we will use shap file from GADM (Global administrative database) to include all the countries at subnational level
#However, the GADM uses its own region naming, so we will need to manually map the names of those regions to GADM code:
#As a first test, we will take Ukraine and look at mapping in the file:
import geopandas as gpd, requests, io

#gadm level 1 for ukraine as a test - gpkg format
url = "https://geodata.ucdavis.edu/gadm/gadm4.1/gpkg/gadm41_UKR.gpkg"
r = requests.get(url)
gdf = gpd.read_file(io.BytesIO(r.content), layer="ADM_ADM_1")

#just print region names so we can build the mapping to our UA codes
print(gdf[["GID_1", "NAME_1"]].to_string(index=False))

In [ ]:
#now, by matching the same names of GADM to our current eurostat codes, we will impute latitude and longitude of all missing places:
import geopandas as gpd
import pandas as pd
import requests, io

#gadm url template
def gadm_url(iso3):
    return f"https://geodata.ucdavis.edu/gadm/gadm4.1/gpkg/gadm41_{iso3}.gpkg"

def get_gadm_centroids(iso3, code_to_name: dict):
    #download gadm level 1 for country and extract centroids for needed regions
    r = requests.get(gadm_url(iso3))
    gdf = gpd.read_file(io.BytesIO(r.content), layer="ADM_ADM_1")
    gdf_proj = gdf.to_crs(epsg=3035)
    gdf["lon"] = gdf_proj.geometry.centroid.to_crs(epsg=4326).x
    gdf["lat"] = gdf_proj.geometry.centroid.to_crs(epsg=4326).y
    #flip dict: name -- code
    name_to_code = {v: k for k, v in code_to_name.items()}
    gdf["reg_nuts2"] = gdf["NAME_1"].map(name_to_code)
    matched = gdf[gdf["reg_nuts2"].notna()][["reg_nuts2", "lon", "lat"]]
    print(f"{iso3}: matched {len(matched)}/{len(code_to_name)} regions")
    return matched

#ukraine
ua_map = {
    "UA11": "Vinnytsya",
    "UA12": "Volyn",
    "UA13": "Dnipropetrovs'k",
    "UA14": "Donets'k",
    "UA21": "Zhytomyr",
    "UA23": "Zakarpattia",
    "UA24": "Zaporizhia",
    "UA3":  "Kiev City",
    "UA31": "Ivano-Frankivs'k",
    "UA41": "Kiev",
    "UA42": "Kirovohrad",
    "UA51": "Luhans'k",
    "UA52": "L'viv",
    "UA53": "Mykolayiv",
    "UA61": "Odessa",
    "UA62": "Poltava",
    "UA63": "Rivne",
    "UA71": "Sumy",
    "UA72": "Ternopil'",
    "UA73": "Kharkiv",
    "UA81": "Kherson",
    "UA82": "Khmel'nyts'kyy",
    "UA83": "Cherkasy",
}

#armenia
am_map = {
    "AM01": "Aragatsotn",
    "AM02": "Ararat",
    "AM03": "Erevan",
    "AM04": "Gegharkunik",
}

#georgia
ge_map = {
    "GE01": "Tbilisi",
    "GE02": "Ajaria",
    "GE03": "Guria",
}

#belarus
by_map = {
    "BY01": "Brest",
    "BY02": "Gomel",
    "BY03": "Grodno",
    "BY04": "Mogilev",
    "BY05": "Minsk", #Note on Belarus BY06: GADM doesn't separate Minsk city from Minsk oblast, so both get the same centroid. That's acceptable since they're geographically the same area.
    "BY06": "Minsk",
    "BY07": "Vitebsk",
}

#bosnia
ba_map = {
    "BABIH": "Federacija Bosna i Hercegovina",
    "BABRC": "Brčko",
    "BASRP": "Repuplika Srpska",
}

#russia
ru_map = {
    "RU11": "Moscow City",
    "RU21": "City of St. Petersburg",
    "RU31": "Leningrad",
    "RU41": "Moskva",
    "RU51": "Krasnodar",
    "RU61": "Sverdlovsk",
    "RU71": "Novosibirsk",
    "RU81": "Tatarstan",
}

#azerbaijan mapping using exact gadm names
az_map = {
    "AZ-ABS": "Absheron",
    "AZ-AGS": "Aran",
    "AZ-AGU": "Aran",
    "AZ-AST": "Lankaran",
    "AZ-BA":  "Absheron",       #baku metro
    "AZ-BAL": "Absheron",
    "AZ-BAR": "Aran",
    "AZ-BEY": "Absheron",       #baku district
    "AZ-BIL": "Absheron",
    "AZ-GA":  "Ganja-Qazakh",
    "AZ-GAD": "Aran",
    "AZ-GOR": "Kalbajar-Lachin",
    "AZ-GOY": "Absheron",
    "AZ-GYG": "Ganja-Qazakh",
    "AZ-HAC": "Aran",
    "AZ-IMI": "Absheron",
    "AZ-KUR": "Aran",
    "AZ-LAN": "Lankaran",
    "AZ-MAS": "Lankaran",
    "AZ-MI":  "Aran",
    "AZ-NEF": "Aran",
    "AZ-QAB": "Quba-Khachmaz",
    "AZ-QAZ": "Ganja-Qazakh",
    "AZ-QUS": "Quba-Khachmaz",
    "AZ-SAB": "Aran",
    "AZ-SAL": "Lankaran",
    "AZ-SAT": "Shaki-Zaqatala",
    "AZ-SKR": "Aran",
    "AZ-SM":  "Shaki-Zaqatala",
    "AZ-SMX": "Shaki-Zaqatala",
    "AZ-TOV": "Absheron",
    "AZ-XAC": "Kalbajar-Lachin",
    "AZ-ZAR": "Daglig-Shirvan",
}

#run all countries with fixed mappings
all_centroids = []
for iso3, label, mapping in [("UKR","ukraine", ua_map), ("ARM","armenia", am_map),
                               ("GEO","georgia", ge_map), ("BLR","belarus", by_map),
                               ("BIH","bosnia", ba_map), ("RUS","russia", ru_map),
                               ("AZE","azerbaijan", az_map)]:
    try:
        result = get_gadm_centroids(iso3, mapping)
        all_centroids.append(result)
        print(f"{label}: matched {len(result)}/{len(set(mapping.values()))} gadm regions -> {len(mapping)} codes")
    except Exception as e:
        print(f"{label} failed: {e}")

#combine all gadm centroids
gadm_centroids = pd.concat(all_centroids).drop_duplicates(subset="reg_nuts2")
print(f"\ntotal gadm centroids built: {len(gadm_centroids)}")

#load our data and patch in gadm centroids for rows still missing lon/lat
df = pd.read_csv("/content/drive/MyDrive/thesis/spatial_clean.csv")

for _, row in gadm_centroids.iterrows():
    mask = (df["reg_nuts2"] == row["reg_nuts2"]) & (df["lon"].isna())
    df.loc[mask, "lon"] = row["lon"]
    df.loc[mask, "lat"] = row["lat"]

#check whats still missing
still_missing = df[df["lon"].isna()]["reg_nuts2"].unique()
print(f"\nstill missing after gadm patch: {len(still_missing)} codes -> {sorted(still_missing)}")

df.to_csv("/content/drive/MyDrive/thesis/spatial_clean.csv", index=False)
print("saved: spatial_clean.csv")

In [ ]:
#print exact gadm names for all countries with mismatches so we can fix mappings
#additionally, we print which codes didnt get matched so we know exactly whats left

for iso3, label, mapping in [("ARM","armenia", am_map), ("GEO","georgia", ge_map),
                               ("BLR","belarus", by_map), ("BIH","bosnia", ba_map),
                               ("RUS","russia", ru_map), ("AZE","azerbaijan", az_map)]:
    r = requests.get(gadm_url(iso3))
    gdf = gpd.read_file(io.BytesIO(r.content), layer="ADM_ADM_1")

    #exact gadm names
    print(f"\n{label} gadm names: {gdf['NAME_1'].tolist()}")

    #which of our codes didnt match
    name_to_code = {v: k for k, v in mapping.items()}
    unmatched_codes = {code: name for code, name in mapping.items() if name not in gdf['NAME_1'].values}
    print(f"{label} unmatched codes ({len(unmatched_codes)}): {unmatched_codes}")

In [ ]:
#even though all Azerbaijan names matched, we still see that not all lon lat were in gadm names and only matched 8 regions out of 33:
#let's inspect why it failed:
#check what az_map values look like vs gadm names to see why it failed
r = requests.get(gadm_url("AZE"))
az_gdf = gpd.read_file(io.BytesIO(r.content), layer="ADM_ADM_1")
print("exact az gadm names:", az_gdf["NAME_1"].tolist())

#also check what get_gadm_centroids returned for az
result_az = get_gadm_centroids("AZE", az_map)
print(result_az)

In [ ]:
#let's then impute all the centroids manually to our file(only 29 regions left):
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/thesis/spatial_clean.csv")

#manual centroids for non-az missing codes
manual = {
    "BY05":  (53.9045, 27.5615),   #minsk city, belarus
    "NL326": (52.3676, 4.9041),    #amsterdam, noord-holland
    "NO0C":  (63.4305, 10.3951),   #trondheim, central norway
    "NO0D":  (59.9139, 10.7522),   #oslo, eastern norway
    #new ukraine codes
    "UA22":  (51.4982, 31.2893),   #chernihiv oblast
    "UA43":  (46.9750, 31.9946),   #mykolaiv oblast
    "UA84":  (44.9521, 34.1024),   #crimea
}

#az districts mapped to gadm economic region centroids
az_manual = {
    "AZ-ABS": (49.435594, 40.454469),   #absheron
    "AZ-AGS": (48.132166, 40.019201),   #aran
    "AZ-AGU": (48.132166, 40.019201),   #aran
    "AZ-AST": (48.569585, 38.921362),   #lankaran
    "AZ-BA":  (49.435594, 40.454469),   #absheron (baku)
    "AZ-BAL": (49.435594, 40.454469),   #absheron
    "AZ-BAR": (48.132166, 40.019201),   #aran
    "AZ-BEY": (49.435594, 40.454469),   #absheron
    "AZ-BIL": (49.435594, 40.454469),   #absheron
    "AZ-GA":  (46.004433, 40.812757),   #ganja-qazakh
    "AZ-GAD": (48.132166, 40.019201),   #aran
    "AZ-GOR": (46.335349, 39.773141),   #kalbajar-lachin
    "AZ-GOY": (49.435594, 40.454469),   #absheron
    "AZ-GYG": (46.004433, 40.812757),   #ganja-qazakh
    "AZ-HAC": (48.132166, 40.019201),   #aran
    "AZ-IMI": (49.435594, 40.454469),   #absheron
    "AZ-KUR": (48.132166, 40.019201),   #aran
    "AZ-LAN": (48.569585, 38.921362),   #lankaran
    "AZ-MAS": (48.569585, 38.921362),   #lankaran
    "AZ-MI":  (48.132166, 40.019201),   #aran
    "AZ-NEF": (48.132166, 40.019201),   #aran
    "AZ-QAB": (48.582694, 41.275773),   #quba-khachmaz
    "AZ-QAZ": (46.004433, 40.812757),   #ganja-qazakh
    "AZ-QUS": (48.582694, 41.275773),   #quba-khachmaz
    "AZ-SAB": (48.132166, 40.019201),   #aran
    "AZ-SAL": (48.569585, 38.921356),   #lankaran
    "AZ-SAT": (47.114261, 41.232062),   #shaki-zaqatala
    "AZ-SKR": (48.132166, 40.019201),   #aran
    "AZ-SM":  (47.114261, 41.232062),   #shaki-zaqatala
    "AZ-SMX": (47.114261, 41.232062),   #shaki-zaqatala
    "AZ-TOV": (49.435594, 40.454469),   #absheron
    "AZ-XAC": (46.335349, 39.773141),   #kalbajar-lachin
    "AZ-ZAR": (48.481434, 40.640631),   #daglig-shirvan
    "AZ-AGA": (45.453890, 41.118890),   #aghstafa-- ganja-qazakh
    "AZ-ISM": (48.150000, 40.780000),   #ismayilli -- daglig-shirvan
    "AZ-QBA": (48.520000, 41.360000),   #quba -- quba-khachmaz
    "AZ-SAK": (47.170000, 41.190000),   #shaki rayon -- shaki-zaqatala
    "AZ-SMI": (48.630000, 40.630000),   #shamakhi -- daglig-shirvan
    "AZ-SR":  (48.920000, 39.940000),   #shirvan city -- aran
}

#patch all in one loop
for code, (lat, lon) in {**manual, **az_manual}.items():
    mask = (df["reg_nuts2"] == code) & (df["lon"].isna())
    df.loc[mask, "lon"] = lon
    df.loc[mask, "lat"] = lat
    if mask.sum() > 0:
        print(f"{code}: imputed {mask.sum()} rows")

#final check
still_missing = df[df["lon"].isna()]["reg_nuts2"].unique()
print(f"\nstill missing: {len(still_missing)} -> {sorted(still_missing)}")

df.to_csv("/content/drive/MyDrive/thesis/spatial_clean.csv", index=False)
print("saved: spatial_clean.csv")

In [ ]:
#Finally, we continue with Moran's I and Lisa clustering
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from libpysal.weights import KNN, lag_spatial
from esda.moran import Moran, Moran_Local
from shapely.geometry import Point
from scipy.stats import norm
from pyproj import Transformer

df = pd.read_csv("/content/drive/MyDrive/thesis/spatial_clean.csv")
df["error"] = (~df["correct"].astype(bool)).astype(int)

# aggregate errors to region level with centroid coordinates
region = (
    df.groupby("reg_nuts2")
    .agg(
        error_rate=("error", "mean"),
        n_obs=("error", "count"),
        lon=("lon", "first"),
        lat=("lat", "first"),
    )
    .reset_index()
)

# build geodataframe from region centroids
gdf = gpd.GeoDataFrame(
    region,
    geometry=[Point(lon, lat) for lon, lat in zip(region["lon"], region["lat"])],
    crs="EPSG:4326"
)

# knn k=5 weights matrix, row standardized
w = KNN.from_dataframe(gdf, k=5)
w.transform = "r"

# global moran's i
moran = Moran(gdf["error_rate"], w)
print(f"regions: {len(region)} | mean error rate: {region['error_rate'].mean():.3f}")
print(f"\nglobal moran's i: {moran.I:.4f}")
print(f"expected i:       {moran.EI:.4f}")
print(f"p-value:          {moran.p_sim:.4f}")
print(f"z-score:          {moran.z_sim:.4f}")
print(f"conclusion:       {'spatial autocorrelation detected' if moran.p_sim < 0.05 else 'no significant spatial autocorrelation'}")

# local moran's i (lisa)
lisa = Moran_Local(gdf["error_rate"], w)
gdf["lisa_p"] = lisa.p_sim
gdf["lisa_q"] = lisa.q

cluster_labels = {1: "High-High", 2: "Low-High", 3: "Low-Low", 4: "High-Low"}
gdf["cluster"] = np.where(
    gdf["lisa_p"] < 0.05,
    gdf["lisa_q"].map(cluster_labels),
    "Not significant"
)

print("\nlisa cluster counts:")
print(gdf["cluster"].value_counts().to_string())
print(f"\nhigh-high regions: {gdf[gdf['cluster']=='High-High']['reg_nuts2'].tolist()}")
print(f"low-low regions:   {gdf[gdf['cluster']=='Low-Low']['reg_nuts2'].tolist()}")

# prep for map plots — project to web mercator and load europe basemap
world = gpd.read_file("https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip")
europe = world.cx[-25:85, 25:85]
gdf_web = gdf.to_crs(epsg=3857)
europe_web = europe.to_crs(epsg=3857)

# bounding box for map zoom
transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
x_min, y_min = transformer.transform(gdf["lon"].min() - 2, gdf["lat"].min() - 2)
x_max, y_max = transformer.transform(gdf["lon"].max() + 2, gdf["lat"].max() + 2)

# moran scatter components
lag = lag_spatial(w, gdf["error_rate"].values)
z  = gdf["error_rate"].values - gdf["error_rate"].mean()
zl = lag - lag.mean()

cluster_colors = {
    "High-High":       "red",
    "Low-Low":         "steelblue",
    "High-Low":        "orange",
    "Low-High":        "lightblue",
    "Not significant": "lightgrey"
}

In [ ]:
#1: bell curve
fig1, ax_bell = plt.subplots(figsize=(12, 5))
sim_values = moran.sim
ax_bell.hist(sim_values, bins=40, color="steelblue", edgecolor="white",
             alpha=0.8, density=True, label="simulated I (999 permutations)")
mu, std = norm.fit(sim_values)
x = np.linspace(min(sim_values), max(sim_values), 200)
ax_bell.plot(x, norm.pdf(x, mu, std), "navy", linewidth=2, label="fitted normal")
ax_bell.axvline(moran.I, color="red", linewidth=2.5, linestyle="--",
                label=f"observed I = {moran.I:.3f}")
crit = np.percentile(sim_values, 95)
ax_bell.axvline(crit, color="orange", linewidth=1.5, linestyle=":",
                label=f"95th percentile = {crit:.3f}")
x_fill = np.linspace(crit, max(sim_values), 100)
ax_bell.fill_between(x_fill, norm.pdf(x_fill, mu, std),
                     alpha=0.25, color="orange", label="rejection region (p<0.05)")
ax_bell.set_title(
    f"Moran's I Permutation Test — Null Distribution\n"
    f"Observed I = {moran.I:.3f} | Expected I = {moran.EI:.4f} | "
    f"z = {moran.z_sim:.2f} | p = {moran.p_sim:.4f}", fontsize=12)
ax_bell.set_xlabel("Moran's I value", fontsize=10)
ax_bell.set_ylabel("density", fontsize=10)
ax_bell.legend(fontsize=9)
plt.tight_layout()
plt.savefig(path("moran_bell.png"), dpi=150, bbox_inches="tight")
plt.show()

#2: moran scatter
fig2, ax_scatter = plt.subplots(figsize=(7, 6))
ax_scatter.scatter(z, zl, c="steelblue", alpha=0.5, edgecolors="grey", linewidths=0.3, s=30)
ax_scatter.axhline(0, color="black", linewidth=0.8)
ax_scatter.axvline(0, color="black", linewidth=0.8)
m, b = np.polyfit(z, zl, 1)
xline = np.linspace(z.min(), z.max(), 100)
ax_scatter.plot(xline, m * xline + b, "red", linewidth=2, label=f"slope = {m:.3f} (≈ Moran's I)")
ax_scatter.set_xlabel("error rate (mean-centered)", fontsize=9)
ax_scatter.set_ylabel("spatial lag of error rate", fontsize=9)
ax_scatter.set_title("Moran Scatter Plot\n(HH=top right, LL=bottom left)", fontsize=10)
ax_scatter.legend(fontsize=8)
ax_scatter.text(0.97, 0.97, "HH", transform=ax_scatter.transAxes, ha="right", va="top", color="red", fontsize=11, fontweight="bold")
ax_scatter.text(0.03, 0.03, "LL", transform=ax_scatter.transAxes, ha="left", va="bottom", color="steelblue", fontsize=11, fontweight="bold")
ax_scatter.text(0.97, 0.03, "HL", transform=ax_scatter.transAxes, ha="right", va="bottom", color="orange", fontsize=11, fontweight="bold")
ax_scatter.text(0.03, 0.97, "LH", transform=ax_scatter.transAxes, ha="left", va="top", color="lightblue", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig(path("moran_scatter.png"), dpi=150, bbox_inches="tight")
plt.show()

#3: choropleth error rate map
fig3, ax_choro = plt.subplots(figsize=(12, 8))
europe_web.plot(ax=ax_choro, color="whitesmoke", edgecolor="grey", linewidth=0.4, zorder=1)
sc = ax_choro.scatter(
    gdf_web.geometry.x, gdf_web.geometry.y,
    c=gdf["error_rate"], cmap="YlOrRd",
    s=gdf["n_obs"] * 0.4, alpha=0.85,
    edgecolors="dimgrey", linewidths=0.3, zorder=2
)
plt.colorbar(sc, ax=ax_choro, label="error rate", shrink=0.7)
ax_choro.set_xlim(x_min, x_max)
ax_choro.set_ylim(y_min, y_max)
ax_choro.set_title(f"LLM Error Rate by NUTS2 Region\nMoran's I = {moran.I:.3f}, p = {moran.p_sim:.3f}", fontsize=12)
ax_choro.set_axis_off()
plt.tight_layout()
plt.savefig(path("moran_choropleth.png"), dpi=150, bbox_inches="tight")
plt.show()

#4: lisa cluster map
fig4, ax_lisa = plt.subplots(figsize=(12, 8))
europe_web.plot(ax=ax_lisa, color="whitesmoke", edgecolor="grey", linewidth=0.4, zorder=1)
for cluster, color in cluster_colors.items():
    mask = gdf["cluster"] == cluster
    if mask.sum() > 0:
        ax_lisa.scatter(
            gdf_web.loc[mask].geometry.x,
            gdf_web.loc[mask].geometry.y,
            c=color, s=gdf.loc[mask, "n_obs"] * 0.4,
            alpha=0.85, edgecolors="dimgrey", linewidths=0.3,
            zorder=2, label=f"{cluster} (n={mask.sum()})"
        )
ax_lisa.set_xlim(x_min, x_max)
ax_lisa.set_ylim(y_min, y_max)
ax_lisa.set_title("LISA Cluster Map", fontsize=12)
ax_lisa.legend(loc="lower left", fontsize=9, framealpha=0.8)
ax_lisa.set_axis_off()
plt.tight_layout()
plt.savefig(path("moran_lisa.png"), dpi=150, bbox_inches="tight")
plt.show()

print("saved: moran_bell.png, moran_scatter.png, moran_choropleth.png, moran_lisa.png")

## 5.b Non-religious spatial analysis

In [ ]:
import pandas as pd

# load both files
spatial = pd.read_csv(path("spatial_clean.csv"))
no_religion = pd.read_csv(path("llm_results_all_selected_gpt-5.1_no_religion.csv"))

print(f"spatial_clean: {spatial.shape[0]:,} rows")
print(f"no_religion results: {no_religion.shape[0]:,} rows")


# align uniqid types
spatial["uniqid"] = spatial["uniqid"].astype(str).str.strip()
no_religion["uniqid"] = no_religion["uniqid"].astype(str).str.strip()

# grab only what we need from spatial_clean — uniqid, lon, lat, cntry, reg_nuts2
geo_slim = spatial[["uniqid", "cntry", "reg_nuts2", "lon", "lat", "gwght"]].drop_duplicates(subset="uniqid")

# merge lon/lat into no_religion results
spatial_no_religion = no_religion.merge(geo_slim, on="uniqid", how="inner")

print(f"spatial_no_religion: {spatial_no_religion.shape[0]:,} rows")
print(f"unmatched dropped: {len(no_religion) - spatial_no_religion.shape[0]:,}")

# save
spatial_no_religion.to_csv(path("spatial_no_religion.csv"), index=False)
print("saved: spatial_no_religion.csv")

In [ ]:
#Finally, we continue with Moran's I and Lisa clustering
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from libpysal.weights import KNN, lag_spatial
from esda.moran import Moran, Moran_Local
from shapely.geometry import Point
from scipy.stats import norm
from pyproj import Transformer

df = pd.read_csv("/content/drive/MyDrive/thesis/spatial_no_religion.csv")
df["error"] = (~df["correct"].astype(bool)).astype(int)

# aggregate errors to region level with centroid coordinates
region = (
    df.groupby("reg_nuts2")
    .agg(
        error_rate=("error", "mean"),
        n_obs=("error", "count"),
        lon=("lon", "first"),
        lat=("lat", "first"),
    )
    .reset_index()
)

# build geodataframe from region centroids
gdf = gpd.GeoDataFrame(
    region,
    geometry=[Point(lon, lat) for lon, lat in zip(region["lon"], region["lat"])],
    crs="EPSG:4326"
)

# knn k=5 weights matrix, row standardized
w = KNN.from_dataframe(gdf, k=5)
w.transform = "r"

# global moran's i
moran = Moran(gdf["error_rate"], w)
print(f"regions: {len(region)} | mean error rate: {region['error_rate'].mean():.3f}")
print(f"\nglobal moran's i: {moran.I:.4f}")
print(f"expected i:       {moran.EI:.4f}")
print(f"p-value:          {moran.p_sim:.4f}")
print(f"z-score:          {moran.z_sim:.4f}")
print(f"conclusion:       {'spatial autocorrelation detected' if moran.p_sim < 0.05 else 'no significant spatial autocorrelation'}")

# local moran's i (lisa)
lisa = Moran_Local(gdf["error_rate"], w)
gdf["lisa_p"] = lisa.p_sim
gdf["lisa_q"] = lisa.q

cluster_labels = {1: "High-High", 2: "Low-High", 3: "Low-Low", 4: "High-Low"}
gdf["cluster"] = np.where(
    gdf["lisa_p"] < 0.05,
    gdf["lisa_q"].map(cluster_labels),
    "Not significant"
)

print("\nlisa cluster counts:")
print(gdf["cluster"].value_counts().to_string())
print(f"\nhigh-high regions: {gdf[gdf['cluster']=='High-High']['reg_nuts2'].tolist()}")
print(f"low-low regions:   {gdf[gdf['cluster']=='Low-Low']['reg_nuts2'].tolist()}")

# prep for map plots — project to web mercator and load europe basemap
world = gpd.read_file("https://naciscdn.org/naturalearth/110m/cultural/ne_110m_admin_0_countries.zip")
europe = world.cx[-25:85, 25:85]
gdf_web = gdf.to_crs(epsg=3857)
europe_web = europe.to_crs(epsg=3857)

# bounding box for map zoom
transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
x_min, y_min = transformer.transform(gdf["lon"].min() - 2, gdf["lat"].min() - 2)
x_max, y_max = transformer.transform(gdf["lon"].max() + 2, gdf["lat"].max() + 2)

# moran scatter components
lag = lag_spatial(w, gdf["error_rate"].values)
z  = gdf["error_rate"].values - gdf["error_rate"].mean()
zl = lag - lag.mean()

cluster_colors = {
    "High-High":       "red",
    "Low-Low":         "steelblue",
    "High-Low":        "orange",
    "Low-High":        "lightblue",
    "Not significant": "lightgrey"
}

In [ ]:
#1: bell curve
fig1, ax_bell = plt.subplots(figsize=(12, 5))
sim_values = moran.sim
ax_bell.hist(sim_values, bins=40, color="steelblue", edgecolor="white",
             alpha=0.8, density=True, label="simulated I (999 permutations)")
mu, std = norm.fit(sim_values)
x = np.linspace(min(sim_values), max(sim_values), 200)
ax_bell.plot(x, norm.pdf(x, mu, std), "navy", linewidth=2, label="fitted normal")
ax_bell.axvline(moran.I, color="red", linewidth=2.5, linestyle="--",
                label=f"observed I = {moran.I:.3f}")
crit = np.percentile(sim_values, 95)
ax_bell.axvline(crit, color="orange", linewidth=1.5, linestyle=":",
                label=f"95th percentile = {crit:.3f}")
x_fill = np.linspace(crit, max(sim_values), 100)
ax_bell.fill_between(x_fill, norm.pdf(x_fill, mu, std),
                     alpha=0.25, color="orange", label="rejection region (p<0.05)")
ax_bell.set_title(
    f"Moran's I Permutation Test — Null Distribution\n"
    f"Observed I = {moran.I:.3f} | Expected I = {moran.EI:.4f} | "
    f"z = {moran.z_sim:.2f} | p = {moran.p_sim:.4f}", fontsize=12)
ax_bell.set_xlabel("Moran's I value", fontsize=10)
ax_bell.set_ylabel("density", fontsize=10)
ax_bell.legend(fontsize=9)
plt.tight_layout()
plt.savefig(path("moran_bell_no_religion.png"), dpi=150, bbox_inches="tight")
plt.show()

#2: moran scatter
fig2, ax_scatter = plt.subplots(figsize=(7, 6))
ax_scatter.scatter(z, zl, c="steelblue", alpha=0.5, edgecolors="grey", linewidths=0.3, s=30)
ax_scatter.axhline(0, color="black", linewidth=0.8)
ax_scatter.axvline(0, color="black", linewidth=0.8)
m, b = np.polyfit(z, zl, 1)
xline = np.linspace(z.min(), z.max(), 100)
ax_scatter.plot(xline, m * xline + b, "red", linewidth=2, label=f"slope = {m:.3f} (≈ Moran's I)")
ax_scatter.set_xlabel("error rate (mean-centered)", fontsize=9)
ax_scatter.set_ylabel("spatial lag of error rate", fontsize=9)
ax_scatter.set_title("Moran Scatter Plot\n(HH=top right, LL=bottom left)", fontsize=10)
ax_scatter.legend(fontsize=8)
ax_scatter.text(0.97, 0.97, "HH", transform=ax_scatter.transAxes, ha="right", va="top", color="red", fontsize=11, fontweight="bold")
ax_scatter.text(0.03, 0.03, "LL", transform=ax_scatter.transAxes, ha="left", va="bottom", color="steelblue", fontsize=11, fontweight="bold")
ax_scatter.text(0.97, 0.03, "HL", transform=ax_scatter.transAxes, ha="right", va="bottom", color="orange", fontsize=11, fontweight="bold")
ax_scatter.text(0.03, 0.97, "LH", transform=ax_scatter.transAxes, ha="left", va="top", color="lightblue", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig(path("moran_scatter_no_religion.png"), dpi=150, bbox_inches="tight")
plt.show()

#3: choropleth error rate map
fig3, ax_choro = plt.subplots(figsize=(12, 8))
europe_web.plot(ax=ax_choro, color="whitesmoke", edgecolor="grey", linewidth=0.4, zorder=1)
sc = ax_choro.scatter(
    gdf_web.geometry.x, gdf_web.geometry.y,
    c=gdf["error_rate"], cmap="YlOrRd",
    s=gdf["n_obs"] * 0.4, alpha=0.85,
    edgecolors="dimgrey", linewidths=0.3, zorder=2
)
plt.colorbar(sc, ax=ax_choro, label="error rate", shrink=0.7)
ax_choro.set_xlim(x_min, x_max)
ax_choro.set_ylim(y_min, y_max)
ax_choro.set_title(f"LLM Error Rate by NUTS2 Region\nMoran's I = {moran.I:.3f}, p = {moran.p_sim:.3f}", fontsize=12)
ax_choro.set_axis_off()
plt.tight_layout()
plt.savefig(path("moran_choropleth_no_religion.png"), dpi=150, bbox_inches="tight")
plt.show()

#4: lisa cluster map
fig4, ax_lisa = plt.subplots(figsize=(12, 8))
europe_web.plot(ax=ax_lisa, color="whitesmoke", edgecolor="grey", linewidth=0.4, zorder=1)
for cluster, color in cluster_colors.items():
    mask = gdf["cluster"] == cluster
    if mask.sum() > 0:
        ax_lisa.scatter(
            gdf_web.loc[mask].geometry.x,
            gdf_web.loc[mask].geometry.y,
            c=color, s=gdf.loc[mask, "n_obs"] * 0.4,
            alpha=0.85, edgecolors="dimgrey", linewidths=0.3,
            zorder=2, label=f"{cluster} (n={mask.sum()})"
        )
ax_lisa.set_xlim(x_min, x_max)
ax_lisa.set_ylim(y_min, y_max)
ax_lisa.set_title("LISA Cluster Map", fontsize=12)
ax_lisa.legend(loc="lower left", fontsize=9, framealpha=0.8)
ax_lisa.set_axis_off()
plt.tight_layout()
plt.savefig(path("moran_lisa_no_religion.png"), dpi=150, bbox_inches="tight")
plt.show()

print("saved: moran_bell_no_religion.png, moran_scatter_no_religion.png, moran_choropleth_no_religion.png, moran_lisa_no_religion.png")

The end!